# MetaMIRAGE Concurrent Preload Worker

This notebook is the **generic one-state worker** for the concurrent MetaMIRAGE preload architecture.

Each copy of this notebook runs on its own 1-GPU Jupyter allocation and processes exactly one state at a time. Multiple workers may run concurrently. All workers:

- use state-local SQLite/canonical persistence;
- coordinate global document and RAG-chunk deduplication through the Preload Coordinator API;
- write embeddings directly to one shared Qdrant server;
- never reset, restore, delete, or snapshot the shared Qdrant build collection;
- emit a state-local `crop_occurrences_state.json` and `state_manifest.json`;
- support crash-safe `resume` behavior.

Global crop merging, cumulative Qdrant validation, and Qdrant snapshot creation belong to the separate **`finalize_wave.ipynb`** workflow.


## 1. Install dependencies

Delta's shared Python environment may be read-only, so packages are installed into a local `.python_packages` directory beside the notebook. The cell deliberately avoids reinstalling PyTorch.

In [ ]:
import sys
from pathlib import Path

PKG_DIR = Path.cwd() / ".python_packages"
PKG_DIR.mkdir(exist_ok=True)

!{sys.executable} -m pip install --no-cache-dir --target "{PKG_DIR}" \
    "huggingface-hub>=0.33.5,<1.0" \
    "transformers>=4.41,<5.0" \
    accelerate \
    bitsandbytes \
    sentence-transformers \
    pdfplumber \
    trafilatura \
    beautifulsoup4 \
    requests \
    tqdm \
    readability-lxml \
    lxml_html_clean \
    qdrant-client==1.18.0 \
    zstandard \
    openpyxl \
    xlrd>=2.0.1 \
    python-dateutil

# Remove local sympy/mpmath copies if pip pulled them into the target directory.
# Delta's shared PyTorch environment carries compatible versions and should own these.
import shutil
for pattern in ["sympy", "sympy-*", "mpmath", "mpmath-*"]:
    for candidate in PKG_DIR.glob(pattern):
        if candidate.is_dir():
            shutil.rmtree(candidate, ignore_errors=True)
        else:
            candidate.unlink(missing_ok=True)

if str(PKG_DIR) not in sys.path:
    sys.path.insert(0, str(PKG_DIR))

print("Local package directory:", PKG_DIR)

## 2. Imports and worker configuration

The five worker notebooks are intended to be identical. For each state run, edit only the configuration block and replace the files in that worker's `input/` directory.


In [ ]:
import csv
import gc
import gzip
import hashlib
import io
import json
import logging
import os
import re
import shutil
import socket
import sqlite3
import threading
import time
import traceback
import uuid
import zipfile

from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, Iterator, List, Optional, Sequence, Set, Tuple
from urllib.parse import urlparse, unquote

import requests
import torch
import zstandard as zstd
import pdfplumber
import trafilatura
import openpyxl
import xlrd

from bs4 import BeautifulSoup
from dateutil import parser as date_parser
from huggingface_hub import login
from readability import Document
from requests.adapters import HTTPAdapter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from urllib3.util.retry import Retry

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    PayloadSchemaType,
    PointStruct,
    VectorParams,
)

logging.getLogger("urllib3.connectionpool").setLevel(logging.ERROR)
logging.getLogger("trafilatura").setLevel(logging.ERROR)

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPUs:", torch.cuda.device_count())

In [ ]:
# ============================================================
# EDIT THIS BLOCK FOR EACH STATE ASSIGNMENT
# ============================================================

# Expected folder shape:
#   preload_pipeline/
#     workers/worker_1/   <- open this notebook here
#       preload.ipynb
#       input/            <- replace PDF/CSV/URL inputs for the assigned state
#     workers/worker_2/
#     ...
#     persistent_state/
#     shared/
#
# If your folder layout differs, edit PIPELINE_ROOT once in each generic worker copy.
WORKER_DIR = Path.cwd().resolve()
PIPELINE_ROOT = (
    WORKER_DIR.parent.parent
    if WORKER_DIR.parent.name.lower() == "workers"
    else WORKER_DIR.parent
).resolve()
INPUT_DIR = WORKER_DIR / "input"
PERSISTENT_STATE_ROOT = PIPELINE_ROOT / "persistent_state"
SHARED_ARTIFACTS_DIR = PIPELINE_ROOT / "shared"

# ---------- Build / wave / state ----------
BUILD_ID = "build_2026_08"
WAVE_ID = "wave_01"
STATE_NAME = "Illinois"
STATE_CODE = "IL"

# Worker-level reruns are resume-only. If a state was processed with semantically
# wrong inputs/configuration, use a corrected/new BUILD_ID instead of trying to
# surgically delete one state's contributions from the shared build.
RUN_MODE = "resume"

# RUN_ID is stable across notebook/node restarts for the same state in the same build.
_safe_build_id = re.sub(r"[^A-Za-z0-9_.-]+", "_", BUILD_ID.strip())
RUN_ID = f"{_safe_build_id}__{STATE_CODE.strip().upper()}"

# WORKER_ID is session-specific. A resumed state on a different node/notebook may
# legitimately have a different WORKER_ID while keeping the same RUN_ID.
WORKER_SLOT = WORKER_DIR.name
WORKER_SESSION_ID = f"{socket.gethostname()}-{os.getpid()}-{uuid.uuid4().hex[:8]}"
WORKER_ID = f"{WORKER_SLOT}:{WORKER_SESSION_ID}"

# ---------- Inputs ----------
# Inputs are AUTO-DISCOVERED directly under INPUT_DIR.
# Filename contracts are case-insensitive:
#   *PDF*.zip                    -> the single PDF ZIP
#   *CSV*.zip                    -> the single CSV ZIP
#   *URL*.txt/.xlsx/.xlsm/.xls  -> the single URL input file
# At most one matching file of each type may exist at a time.
URL_FILE: Optional[Path] = None
PDF_DIR: Optional[Path] = None
PDF_ZIP_FILE: Optional[Path] = None
CSV_ZIP_FILE: Optional[Path] = None
CSV_INPUTS: List[Dict[str, Any]] = []

# Shared support artifacts. The crop file is READ-ONLY from worker notebooks.
# finalize_wave.ipynb is the only component that updates the global crop file.
GLOBAL_CROP_OCCURRENCE_JSON: Path = SHARED_ARTIFACTS_DIR / "crop_occurrences.json"
HARDINESS_CSV: Path = SHARED_ARTIFACTS_DIR / "county_state_hardiness_zone.csv"

# ---------- Shared services ----------
# Override with environment variables when the service-node hostname changes.
COORDINATOR_URL = os.environ.get(
    "METAMIRAGE_COORDINATOR_URL",
    "http://127.0.0.1:8001",
).rstrip("/")
QDRANT_URL = os.environ.get(
    "QDRANT_URL",
    "http://127.0.0.1:6333",
).rstrip("/")
QDRANT_API_KEY: Optional[str] = os.environ.get("QDRANT_API_KEY") or None
QDRANT_COLLECTION = "mirage_base_build"

# The collection is a build-level global resource. Workers validate it but do not
# create/reset/restore/delete it. Initialize it on the service node before a wave.
WORKER_MAY_CREATE_QDRANT_COLLECTION = False

# ---------- Coordinator leases ----------
STATE_LEASE_SECONDS = 1800
STATE_HEARTBEAT_SECONDS = 60
CONTENT_CLAIM_LEASE_SECONDS = 1800
COORDINATOR_TIMEOUT_SECONDS = 30
COORDINATOR_MAX_RETRIES = 5
COORDINATOR_RETRY_BACKOFF_SECONDS = 2.0

# ---------- Models ----------
CLASSIFIER_MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"

# Prefer environment variable HF_TOKEN. If absent, model-loading cell prompts.
HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()

# ---------- Qualification ----------
QUALIFICATION_CHUNK_CHARS = 7000
QUALIFICATION_OVERLAP_CHARS = 700
MAX_QUALIFICATION_CHUNKS = 20
EARLY_STOP_TOTAL_ENTITIES = 5

# Initial attempt + up to 2 retries = max 3 attempts.
MAX_STAGE_RETRIES = 2

# ---------- Runtime-compatible RAG chunking ----------
RAG_CHUNK_SIZE = 480
RAG_CHUNK_OVERLAP = 80
RAG_HARD_CAP = 512

# ---------- Batching / concurrency inside this one-state worker ----------
EXTRACTION_WORKERS = 8
EMBED_BATCH_SIZE = 64
QDRANT_UPSERT_BATCH_SIZE = 128

# ---------- Versioned cache / pipeline contracts ----------
EXTRACTOR_VERSION = "canonical-extractor-v1"
CLASSIFIER_VERSION = "llama-qualification-v1"
CHUNKER_VERSION = "bge-480-80-v1"
METADATA_CONTRACT_VERSION = "runtime-canonical-v1"
WORKER_PIPELINE_VERSION = "concurrent-preload-worker-v1"

# Optional manual retrieval smoke checks; wave finalizer owns required Qdrant validation.
VALIDATION_QUERIES: List[str] = []

# ---------- Safety / development ----------
# None = process every discovered source.
DEBUG_SOURCE_LIMIT: Optional[int] = None

# Keep False until the configuration above and input/ folder have been reviewed.
RUN_PIPELINE = False

print("RUN_ID:", RUN_ID)
print("BUILD_ID:", BUILD_ID)
print("WAVE_ID:", WAVE_ID)
print("Worker:", WORKER_ID)
print("Worker directory:", WORKER_DIR)
print("Pipeline root:", PIPELINE_ROOT)
print("Input directory:", INPUT_DIR)
print("Coordinator:", COORDINATOR_URL)
print("Qdrant:", QDRANT_URL)


## 3. State-local directory layout and configuration validation

Each state has its own durable processing directory under `persistent_state/<BUILD_ID>/<STATE_CODE>/`. Generic worker folders are reusable execution slots and do not own durable state.

The shared global crop JSON is read-only from workers. Qdrant and coordinator state live outside the worker notebook.


In [ ]:
STATE_KEY = re.sub(r"\s+", " ", STATE_NAME.strip().lower())
STATE_CODE = STATE_CODE.strip().upper()
BUILD_ID = BUILD_ID.strip()
WAVE_ID = WAVE_ID.strip()
RUN_MODE = RUN_MODE.strip().lower()

if not BUILD_ID:
    raise ValueError("BUILD_ID must be non-empty.")
if not WAVE_ID:
    raise ValueError("WAVE_ID must be non-empty.")
if not re.fullmatch(r"[A-Z]{2}", STATE_CODE):
    raise ValueError(f"STATE_CODE must be a two-letter code, got {STATE_CODE!r}")
if RUN_MODE != "resume":
    raise ValueError(
        "Concurrent workers currently support RUN_MODE='resume' only. "
        "For a semantic/correctness rebuild, use a corrected/new BUILD_ID."
    )

INPUT_DIR.mkdir(parents=True, exist_ok=True)
PERSISTENT_STATE_ROOT.mkdir(parents=True, exist_ok=True)
SHARED_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

STATE_DIR = PERSISTENT_STATE_ROOT / BUILD_ID / STATE_CODE
CANONICAL_DIR = STATE_DIR / "canonical"
STATE_DB_PATH = STATE_DIR / "pipeline_state.db"
STATE_CROP_JSON = STATE_DIR / "crop_occurrences_state.json"
STATE_MANIFEST_PATH = STATE_DIR / "state_manifest.json"
INPUT_STAGING_DIR = STATE_DIR / "input_staging"

for path in [STATE_DIR, CANONICAL_DIR, INPUT_STAGING_DIR]:
    path.mkdir(parents=True, exist_ok=True)

def _find_single_input(keyword: str, suffixes: Sequence[str]) -> Optional[Path]:
    keyword = keyword.lower()
    suffixes = {suffix.lower() for suffix in suffixes}

    candidates = sorted(
        [
            path
            for path in INPUT_DIR.iterdir()
            if path.is_file()
            and keyword in path.name.lower()
            and path.suffix.lower() in suffixes
        ],
        key=lambda path: path.name.lower(),
    )

    if len(candidates) > 1:
        raise RuntimeError(
            f"Expected at most one {keyword.upper()} input under {INPUT_DIR}, "
            f"but found {len(candidates)}: {[p.name for p in candidates]}"
        )

    return candidates[0] if candidates else None

# Auto-discover the current worker assignment inputs.
PDF_ZIP_FILE = _find_single_input("PDF", {".zip"})
CSV_ZIP_FILE = _find_single_input("CSV", {".zip"})
URL_FILE = _find_single_input("URL", {".txt", ".xlsx", ".xlsm", ".xls"})
PDF_DIR = None

# Extract every CSV in the single CSV ZIP, if present. Extraction is durable in
# this state's persistent directory so a notebook/node restart can resume safely.
CSV_INPUTS = []
if CSV_ZIP_FILE is not None:
    csv_extracted_dir = INPUT_STAGING_DIR / "csv_zip_extracted"
    csv_extracted_dir.mkdir(parents=True, exist_ok=True)
    marker = csv_extracted_dir / ".extracted_complete"

    if not marker.exists():
        with zipfile.ZipFile(CSV_ZIP_FILE, "r") as zf:
            zf.extractall(csv_extracted_dir)
        marker.write_text(
            datetime.now(timezone.utc).isoformat(),
            encoding="utf-8",
        )

    csv_paths = sorted(
        path
        for path in csv_extracted_dir.rglob("*")
        if path.is_file() and path.suffix.lower() == ".csv"
    )

    if not csv_paths:
        raise RuntimeError(f"No CSV files found inside {CSV_ZIP_FILE}")

    CSV_INPUTS = [{"path": path} for path in csv_paths]

if not HARDINESS_CSV.exists():
    raise FileNotFoundError(f"Hardiness mapping not found: {HARDINESS_CSV}")
if not GLOBAL_CROP_OCCURRENCE_JSON.exists():
    raise FileNotFoundError(
        "Global crop occurrence JSON not found. Workers read the maintained global "
        f"copy from: {GLOBAL_CROP_OCCURRENCE_JSON}"
    )

for cfg in CSV_INPUTS:
    cfg["path"] = Path(cfg["path"])
    if not cfg["path"].exists():
        raise FileNotFoundError(f"CSV input not found: {cfg['path']}")

print("State persistent directory:", STATE_DIR)
print("State ledger:", STATE_DB_PATH)
print("State canonical store:", CANONICAL_DIR)
print("State crop output:", STATE_CROP_JSON)
print("URL file:", URL_FILE)
print("PDF zip:", PDF_ZIP_FILE)
print("CSV zip:", CSV_ZIP_FILE)
print("CSV inputs:", [str(x["path"]) for x in CSV_INPUTS])
print("Global crop occurrence JSON (read-only):", GLOBAL_CROP_OCCURRENCE_JSON)


## 4. State-local SQLite processing ledger

The worker ledger is private to one `(BUILD_ID, STATE_CODE)` and persists across notebook/node restarts. It is **not** shared across states.

Cross-state coordination lives in the Preload Coordinator service, not in this SQLite database.


In [ ]:
def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def get_db() -> sqlite3.Connection:
    conn = sqlite3.connect(STATE_DB_PATH, timeout=60)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    conn.execute("PRAGMA busy_timeout = 60000")
    conn.execute("PRAGMA synchronous = NORMAL")
    return conn

SCHEMA_SQL = r"""
CREATE TABLE IF NOT EXISTS runs (
    run_id TEXT PRIMARY KEY,
    build_id TEXT NOT NULL,
    wave_id TEXT NOT NULL,
    state_name TEXT NOT NULL,
    state_code TEXT NOT NULL,
    run_mode TEXT NOT NULL,
    worker_id TEXT,
    coordinator_url TEXT NOT NULL,
    qdrant_url TEXT NOT NULL,
    qdrant_collection TEXT NOT NULL,
    input_fingerprint_json TEXT,
    status TEXT NOT NULL,
    started_at TEXT,
    completed_at TEXT,
    max_stage_retries INTEGER NOT NULL DEFAULT 2,
    manifest_path TEXT,
    error TEXT
);

CREATE TABLE IF NOT EXISTS source_tasks (
    source_key TEXT PRIMARY KEY,
    run_id TEXT NOT NULL,
    source_type TEXT NOT NULL,
    source_uri TEXT NOT NULL,
    source_payload_json TEXT,
    status TEXT NOT NULL,
    attempt_count INTEGER NOT NULL DEFAULT 0,
    document_id TEXT,
    duplicate INTEGER NOT NULL DEFAULT 0,
    last_error TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id)
);

CREATE INDEX IF NOT EXISTS idx_source_tasks_run_status
ON source_tasks(run_id, status);

CREATE TABLE IF NOT EXISTS documents (
    document_id TEXT PRIMARY KEY,
    content_hash TEXT NOT NULL UNIQUE,
    raw_hash TEXT,
    source_type TEXT NOT NULL,
    canonical_text_path TEXT NOT NULL,
    canonical_metadata_path TEXT NOT NULL,
    canonical_text_chars INTEGER,
    canonical_text_bytes INTEGER,
    language TEXT,
    extractor_version TEXT NOT NULL,
    extraction_status TEXT NOT NULL,
    first_seen_run_id TEXT NOT NULL,
    first_seen_state TEXT NOT NULL,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(first_seen_run_id) REFERENCES runs(run_id)
);

CREATE INDEX IF NOT EXISTS idx_documents_raw_hash
ON documents(raw_hash);

CREATE TABLE IF NOT EXISTS run_documents (
    run_id TEXT NOT NULL,
    document_id TEXT NOT NULL,
    source_type TEXT NOT NULL,
    source_uri TEXT,
    discovery_order INTEGER,
    duplicate INTEGER NOT NULL DEFAULT 0,
    document_status TEXT NOT NULL,
    qualification_status TEXT,
    accepted INTEGER,
    qualification_tag TEXT,
    qualification_subtags_json TEXT,
    qualification_entities_json TEXT,
    qualification_reason TEXT,
    classifier_version TEXT,
    rag_status TEXT,
    qdrant_status TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    PRIMARY KEY(run_id, document_id),
    FOREIGN KEY(run_id) REFERENCES runs(run_id),
    FOREIGN KEY(document_id) REFERENCES documents(document_id)
);

CREATE INDEX IF NOT EXISTS idx_run_documents_run_status
ON run_documents(run_id, document_status);

CREATE TABLE IF NOT EXISTS qualification_chunks (
    qualification_chunk_id TEXT PRIMARY KEY,
    run_id TEXT NOT NULL,
    document_id TEXT NOT NULL,
    chunk_index INTEGER NOT NULL,
    chunk_hash TEXT NOT NULL,
    status TEXT NOT NULL,
    classification_result_json TEXT,
    attempt_count INTEGER NOT NULL DEFAULT 0,
    last_error TEXT,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id),
    FOREIGN KEY(document_id) REFERENCES documents(document_id),
    UNIQUE(run_id, document_id, chunk_index)
);

CREATE INDEX IF NOT EXISTS idx_qchunks_run_status
ON qualification_chunks(run_id, status);

CREATE TABLE IF NOT EXISTS rag_chunks (
    rag_chunk_id TEXT PRIMARY KEY,
    run_id TEXT NOT NULL,
    document_id TEXT NOT NULL,
    page INTEGER NOT NULL,
    chunk_index INTEGER NOT NULL,
    chunk_hash TEXT NOT NULL,
    token_count INTEGER NOT NULL,
    chunker_version TEXT NOT NULL,
    metadata_json TEXT,
    status TEXT NOT NULL,
    embedding_status TEXT NOT NULL,
    qdrant_status TEXT NOT NULL,
    metadata_attempt_count INTEGER NOT NULL DEFAULT 0,
    embedding_attempt_count INTEGER NOT NULL DEFAULT 0,
    qdrant_attempt_count INTEGER NOT NULL DEFAULT 0,
    failure_stage TEXT,
    last_error TEXT,
    qdrant_point_id TEXT NOT NULL,
    created_at TEXT NOT NULL,
    updated_at TEXT NOT NULL,
    FOREIGN KEY(run_id) REFERENCES runs(run_id),
    FOREIGN KEY(document_id) REFERENCES documents(document_id),
    UNIQUE(document_id, page, chunk_index, chunker_version)
);

CREATE INDEX IF NOT EXISTS idx_rag_chunks_run_status
ON rag_chunks(run_id, status);

CREATE INDEX IF NOT EXISTS idx_rag_chunks_hash
ON rag_chunks(chunk_hash);

CREATE TABLE IF NOT EXISTS attempts (
    attempt_id INTEGER PRIMARY KEY AUTOINCREMENT,
    run_id TEXT NOT NULL,
    unit_type TEXT NOT NULL,
    unit_id TEXT NOT NULL,
    stage TEXT NOT NULL,
    attempt_number INTEGER NOT NULL,
    status TEXT NOT NULL,
    started_at TEXT NOT NULL,
    completed_at TEXT,
    error_type TEXT,
    error_message TEXT
);

CREATE INDEX IF NOT EXISTS idx_attempts_run
ON attempts(run_id);

CREATE TABLE IF NOT EXISTS classification_cache (
    cache_key TEXT PRIMARY KEY,
    model_id TEXT NOT NULL,
    classifier_version TEXT NOT NULL,
    state_key TEXT NOT NULL,
    crop_hash TEXT NOT NULL,
    content_hash TEXT NOT NULL,
    validated_output_json TEXT NOT NULL,
    raw_output TEXT,
    created_at TEXT NOT NULL
);
"""

with get_db() as conn:
    conn.execute("PRAGMA journal_mode = WAL")
    conn.executescript(SCHEMA_SQL)

    existing = conn.execute("SELECT * FROM runs WHERE run_id=?", (RUN_ID,)).fetchone()
    if existing is None:
        conn.execute(
            """
            INSERT INTO runs(
                run_id, build_id, wave_id, state_name, state_code,
                run_mode, worker_id, coordinator_url, qdrant_url,
                qdrant_collection, status, started_at, max_stage_retries
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'pending', ?, ?)
            """,
            (
                RUN_ID, BUILD_ID, WAVE_ID, STATE_NAME, STATE_CODE,
                RUN_MODE, WORKER_ID, COORDINATOR_URL, QDRANT_URL,
                QDRANT_COLLECTION, utc_now(), MAX_STAGE_RETRIES,
            ),
        )
    else:
        if existing["build_id"] != BUILD_ID or existing["state_code"] != STATE_CODE:
            raise RuntimeError(
                f"RUN_ID {RUN_ID} already exists with different build/state metadata."
            )
        if existing["wave_id"] != WAVE_ID:
            raise RuntimeError(
                f"State {STATE_CODE} in build {BUILD_ID} is already bound to wave "
                f"{existing['wave_id']!r}; current config says {WAVE_ID!r}."
            )
        conn.execute(
            """
            UPDATE runs
            SET worker_id=?, run_mode=?, coordinator_url=?, qdrant_url=?,
                qdrant_collection=?
            WHERE run_id=?
            """,
            (
                WORKER_ID, RUN_MODE, COORDINATOR_URL, QDRANT_URL,
                QDRANT_COLLECTION, RUN_ID,
            ),
        )

print("State-local SQLite schema ready.")


## 5. Common hashing, state-local canonical store, and attempt helpers

Canonical files are persisted only inside this state's durable directory. Cross-state ownership is decided by the coordinator before a new canonical document is committed.


In [ ]:
UUID_NAMESPACE = uuid.UUID("8d36fffe-8bed-4f98-9d6e-8a53e2755a91")

def normalize_hash_text(text: str) -> str:
    return " ".join((text or "").lower().split())

def stable_hash(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()

def content_hash(text: str) -> str:
    return stable_hash(normalize_hash_text(text))

def bytes_sha256(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def canonical_paths(document_id: str) -> Tuple[Path, Path]:
    shard = document_id[:2]
    doc_dir = CANONICAL_DIR / shard / document_id
    return doc_dir / "content.txt.zst", doc_dir / "metadata.json"

CANONICAL_PERSIST_LOCK = threading.Lock()

def write_zstd_text(path: Path, text: str):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + f".{uuid.uuid4().hex}.tmp")
    compressor = zstd.ZstdCompressor(level=6)
    try:
        with open(tmp, "wb") as raw:
            with compressor.stream_writer(raw) as writer:
                writer.write(text.encode("utf-8"))
        tmp.replace(path)
    finally:
        tmp.unlink(missing_ok=True)

def read_zstd_text(path: Path) -> str:
    decompressor = zstd.ZstdDecompressor()
    with open(path, "rb") as raw:
        with decompressor.stream_reader(raw) as reader:
            return reader.read().decode("utf-8")

def write_json_atomic(path: Path, obj: Any):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    tmp.replace(path)

def load_json(path: Path, default=None):
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def deterministic_point_uuid(logical_id: str) -> str:
    return str(uuid.uuid5(UUID_NAMESPACE, logical_id))

def record_attempt(
    unit_type: str,
    unit_id: str,
    stage: str,
    attempt_number: int,
    status: str,
    started_at: str,
    error: Optional[BaseException] = None,
):
    error_type = type(error).__name__ if error else None
    error_message = str(error)[:4000] if error else None
    with get_db() as conn:
        conn.execute(
            """
            INSERT INTO attempts(
                run_id, unit_type, unit_id, stage, attempt_number,
                status, started_at, completed_at, error_type, error_message
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                RUN_ID, unit_type, unit_id, stage, attempt_number,
                status, started_at, utc_now(), error_type, error_message,
            ),
        )

print("Persistence helpers ready.")

## Coordinator client, state lease, and global dedupe claims

The coordinator is the worker control plane. It owns state leases and atomic build-level dedupe claims. The worker still performs all extraction, qualification, chunking, embedding, and direct Qdrant ingestion itself.

This worker uses the same claim API for two scopes:

- `document`: canonical document `content_hash` ownership;
- `rag_chunk`: global RAG chunk-text hash ownership, preserving the current notebook's build-level chunk deduplication semantics.


In [ ]:
_STATE_HEARTBEAT_STOP = threading.Event()
_STATE_LEASE_LOST = threading.Event()
_STATE_HEARTBEAT_THREAD: Optional[threading.Thread] = None
_STATE_LOCK_OWNED = False
_STATE_ALREADY_COMPLETE = False

def _coordinator_request(
    method: str,
    path: str,
    payload: Optional[Dict[str, Any]] = None,
    *,
    timeout: Optional[float] = None,
) -> Dict[str, Any]:
    url = f"{COORDINATOR_URL}{path}"
    timeout = timeout or COORDINATOR_TIMEOUT_SECONDS
    last_exc: Optional[BaseException] = None

    for attempt in range(COORDINATOR_MAX_RETRIES):
        try:
            response = requests.request(
                method.upper(),
                url,
                json=payload,
                timeout=timeout,
            )
            response.raise_for_status()
            if not response.content:
                return {}
            data = response.json()
            if not isinstance(data, dict):
                raise RuntimeError(f"Coordinator returned non-object JSON from {path}: {data!r}")
            return data
        except Exception as exc:
            last_exc = exc
            if attempt + 1 >= COORDINATOR_MAX_RETRIES:
                break
            time.sleep(COORDINATOR_RETRY_BACKOFF_SECONDS * (attempt + 1))

    raise RuntimeError(
        f"Coordinator request failed after {COORDINATOR_MAX_RETRIES} attempts: "
        f"{method.upper()} {url}"
    ) from last_exc

def coordinator_health() -> Dict[str, Any]:
    return _coordinator_request("GET", "/health")

def _normalized_status(data: Dict[str, Any]) -> str:
    return str(data.get("status") or "").strip().lower()

def coordinator_acquire_state() -> Dict[str, Any]:
    global _STATE_LOCK_OWNED, _STATE_ALREADY_COMPLETE
    data = _coordinator_request(
        "POST",
        "/state/acquire",
        {
            "build_id": BUILD_ID,
            "wave_id": WAVE_ID,
            "state_code": STATE_CODE,
            "state_name": STATE_NAME,
            "worker_id": WORKER_ID,
            "lease_seconds": STATE_LEASE_SECONDS,
        },
    )
    status = _normalized_status(data)

    if status in {"acquired", "already_owned", "renewed"}:
        _STATE_LOCK_OWNED = True
        _STATE_ALREADY_COMPLETE = False
        return data
    if status == "already_complete":
        _STATE_LOCK_OWNED = False
        _STATE_ALREADY_COMPLETE = True
        return data

    raise RuntimeError(
        f"Could not acquire state lease for {BUILD_ID}/{STATE_CODE}: {data}"
    )

def coordinator_state_heartbeat() -> Dict[str, Any]:
    return _coordinator_request(
        "POST",
        "/state/heartbeat",
        {
            "build_id": BUILD_ID,
            "wave_id": WAVE_ID,
            "state_code": STATE_CODE,
            "worker_id": WORKER_ID,
            "lease_seconds": STATE_LEASE_SECONDS,
        },
    )

def _state_heartbeat_loop():
    while not _STATE_HEARTBEAT_STOP.wait(STATE_HEARTBEAT_SECONDS):
        try:
            data = coordinator_state_heartbeat()
            status = _normalized_status(data)
            if status in {"complete", "completed", "already_complete"}:
                return
            if status not in {"renewed", "active", "already_owned", "acquired"}:
                print("STATE LEASE LOST:", data)
                _STATE_LEASE_LOST.set()
                return
        except Exception as exc:
            # A transient network failure does not prove ownership was lost. The
            # next successful heartbeat will explicitly tell us if the lease expired.
            print("Coordinator heartbeat warning:", exc)

def start_state_heartbeat():
    global _STATE_HEARTBEAT_THREAD
    if not _STATE_LOCK_OWNED:
        return
    _STATE_HEARTBEAT_STOP.clear()
    _STATE_LEASE_LOST.clear()
    _STATE_HEARTBEAT_THREAD = threading.Thread(
        target=_state_heartbeat_loop,
        name=f"state-heartbeat-{STATE_CODE}",
        daemon=True,
    )
    _STATE_HEARTBEAT_THREAD.start()

def stop_state_heartbeat():
    _STATE_HEARTBEAT_STOP.set()
    thread = _STATE_HEARTBEAT_THREAD
    if thread and thread.is_alive():
        thread.join(timeout=5)

def assert_state_lease_healthy():
    if _STATE_LEASE_LOST.is_set():
        raise RuntimeError(
            f"Coordinator state lease for {BUILD_ID}/{STATE_CODE} was lost. "
            "Stop this worker and resume only after acquiring the state again."
        )

def coordinator_release_state(reason: str = "worker_release"):
    global _STATE_LOCK_OWNED
    if not _STATE_LOCK_OWNED:
        return
    try:
        _coordinator_request(
            "POST",
            "/state/release",
            {
                "build_id": BUILD_ID,
                "wave_id": WAVE_ID,
                "state_code": STATE_CODE,
                "worker_id": WORKER_ID,
                "reason": reason,
            },
        )
    finally:
        _STATE_LOCK_OWNED = False

def coordinator_mark_state_complete(manifest_path: Path, manifest_sha256: str) -> Dict[str, Any]:
    global _STATE_LOCK_OWNED, _STATE_ALREADY_COMPLETE
    data = _coordinator_request(
        "POST",
        "/state/complete",
        {
            "build_id": BUILD_ID,
            "wave_id": WAVE_ID,
            "state_code": STATE_CODE,
            "state_name": STATE_NAME,
            "worker_id": WORKER_ID,
            "manifest_path": str(manifest_path),
            "manifest_sha256": manifest_sha256,
        },
    )
    status = _normalized_status(data)
    if status not in {"complete", "completed", "already_complete"}:
        raise RuntimeError(f"Coordinator refused state completion: {data}")
    _STATE_LOCK_OWNED = False
    _STATE_ALREADY_COMPLETE = True
    return data

def coordinator_claim_content(
    scope: str,
    hash_value: str,
    *,
    resource_id: Optional[str] = None,
) -> Dict[str, Any]:
    if scope not in {"document", "rag_chunk"}:
        raise ValueError(f"Unsupported claim scope: {scope}")
    data = _coordinator_request(
        "POST",
        "/content/claim",
        {
            "build_id": BUILD_ID,
            "wave_id": WAVE_ID,
            "state_code": STATE_CODE,
            "worker_id": WORKER_ID,
            "scope": scope,
            "content_hash": hash_value,
            "resource_id": resource_id,
            "lease_seconds": CONTENT_CLAIM_LEASE_SECONDS,
        },
    )
    status = _normalized_status(data)
    if status not in {
        "claimed", "already_owned", "already_complete", "claimed_by_other"
    }:
        raise RuntimeError(f"Unexpected coordinator claim response: {data}")
    return data

def coordinator_heartbeat_content(scope: str, hash_value: str) -> Dict[str, Any]:
    return _coordinator_request(
        "POST",
        "/content/heartbeat",
        {
            "build_id": BUILD_ID,
            "wave_id": WAVE_ID,
            "state_code": STATE_CODE,
            "worker_id": WORKER_ID,
            "scope": scope,
            "content_hash": hash_value,
            "lease_seconds": CONTENT_CLAIM_LEASE_SECONDS,
        },
    )

def coordinator_complete_content(
    scope: str,
    hash_value: str,
    *,
    resource_id: str,
) -> Dict[str, Any]:
    data = _coordinator_request(
        "POST",
        "/content/complete",
        {
            "build_id": BUILD_ID,
            "wave_id": WAVE_ID,
            "state_code": STATE_CODE,
            "worker_id": WORKER_ID,
            "scope": scope,
            "content_hash": hash_value,
            "resource_id": resource_id,
        },
    )
    status = _normalized_status(data)
    if status not in {"complete", "completed", "already_complete"}:
        raise RuntimeError(f"Coordinator refused content completion: {data}")
    return data

def coordinator_release_content(scope: str, hash_value: str, reason: str):
    try:
        _coordinator_request(
            "POST",
            "/content/release",
            {
                "build_id": BUILD_ID,
                "wave_id": WAVE_ID,
                "state_code": STATE_CODE,
                "worker_id": WORKER_ID,
                "scope": scope,
                "content_hash": hash_value,
                "reason": reason,
            },
        )
    except Exception as exc:
        # Leases guarantee eventual recovery even if release cannot be delivered.
        print(f"Coordinator release warning ({scope}/{hash_value[:12]}): {exc}")

print("Coordinator client helpers ready.")


## 6. Hardiness-zone contract

This independently reconstructs the inference mapping contract from `county_state_hardiness_zone.csv`.

Resolution order:

1. `(county, state) → exact zone`
2. otherwise `state → modal zone`
3. unrecognized state → `""`

For Alaska and Hawaii only, `hardiness_zone=""` is an explicitly allowed metadata-contract exception.

In [ ]:
def normalize_token(value: Any) -> str:
    value = re.sub(r"[^a-z0-9\s]", " ", str(value or "").lower())
    return re.sub(r"\s+", " ", value).strip()

def normalize_county(value: Any) -> str:
    county = normalize_token(value)
    return re.sub(r"\bcounty\b$", "", county).strip()

def load_hardiness_lookup(csv_path: Path):
    county_lookup: Dict[Tuple[str, str], str] = {}
    state_zone_counts: Dict[str, Counter] = defaultdict(Counter)
    abbr_to_name: Dict[str, str] = {}

    with open(csv_path, newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            state_name = re.sub(r"\s+", " ", (row.get("state") or "").strip())
            state_abbr = (row.get("state_abbr") or "").strip().upper()
            county = normalize_county(row.get("county", ""))
            zone = (row.get("hardiness_zone") or "").strip()

            if state_name and state_abbr:
                abbr_to_name[state_abbr] = state_name.upper()

            if not state_name:
                continue

            canonical_name = state_name.upper()

            if county and zone:
                county_lookup[(county, canonical_name)] = zone

            if zone:
                state_zone_counts[canonical_name][zone] += 1

    state_modal_lookup = {
        state: counts.most_common(1)[0][0]
        for state, counts in state_zone_counts.items()
        if counts
    }

    name_to_abbr = {name: abbr for abbr, name in abbr_to_name.items()}
    return county_lookup, state_modal_lookup, abbr_to_name, name_to_abbr

(
    COUNTY_ZONE_LOOKUP,
    STATE_MODAL_ZONE_LOOKUP,
    STATE_ABBR_TO_NAME,
    STATE_NAME_TO_ABBR,
) = load_hardiness_lookup(HARDINESS_CSV)

def canonical_state(value: Any) -> str:
    state = normalize_token(value).upper()
    if state in STATE_ABBR_TO_NAME:
        return STATE_ABBR_TO_NAME[state]
    if state in STATE_NAME_TO_ABBR:
        return state
    return ""

def hardiness_zone_for_location(location: str) -> str:
    if not location or not location.strip():
        return ""

    parts = [part.strip() for part in location.split(",") if part.strip()]
    if not parts:
        return ""

    if len(parts) >= 2:
        # Preferred: State, County
        state = canonical_state(parts[0])
        county = normalize_county(parts[1])
        if state:
            return (
                COUNTY_ZONE_LOOKUP.get((county, state))
                or STATE_MODAL_ZONE_LOOKUP.get(state, "")
            )

        # Legacy: County, State
        county = normalize_county(parts[0])
        state = canonical_state(parts[1])
        if state:
            return (
                COUNTY_ZONE_LOOKUP.get((county, state))
                or STATE_MODAL_ZONE_LOOKUP.get(state, "")
            )
        return ""

    state = canonical_state(parts[0])
    return STATE_MODAL_ZONE_LOOKUP.get(state, "")

resolved_state = canonical_state(STATE_CODE) or canonical_state(STATE_NAME)
if not resolved_state:
    raise ValueError(f"State is not recognized by hardiness mapping: {STATE_NAME} / {STATE_CODE}")

print("Canonical state:", resolved_state)
print("State modal zone:", hardiness_zone_for_location(STATE_NAME) or "<empty>")
if STATE_CODE in {"AK", "HI"}:
    print("AK/HI exception active: empty hardiness_zone is allowed.")

## 7. Crop occurrence input + state-local crop output helpers

Workers read the maintained global `crop_occurrences.json` as the qualification seed but never modify it. Each worker writes only `crop_occurrences_state.json` under its state directory. `finalize_wave.ipynb` later merges the expected wave states into the one cumulative global crop file.


In [ ]:
ALLOWED_TAGS = {"crops", "pest", "disease", "management", "multi", "msc"}
ENTITY_FIELDS = ["disease", "pests", "management"]

def normalize_name(value: Any) -> str:
    return re.sub(r"\s+", " ", str(value or "").strip().lower())

def normalize_entity(value: Any) -> Optional[str]:
    if value is None:
        return None
    v = normalize_name(value)
    if len(v) < 3:
        return None
    generic_rejects = {
        "disease", "diseases", "pest", "pests", "issue", "issues",
        "problem", "problems", "management", "control", "crop", "crops",
        "plant", "plants", "unknown", "none", "n/a", "na"
    }
    return None if v in generic_rejects else v

def load_crop_occurrence_state(
    path: Optional[Path],
    state_key: str,
) -> Dict[str, Dict[str, Any]]:
    if path is None:
        print("No crop occurrence JSON configured; crop list will start empty.")
        return {}

    raw = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(raw, dict):
        raise ValueError("crop_occurrences.json must contain a top-level JSON object.")

    normalized_state_keys = {
        normalize_name(key): key
        for key in raw.keys()
    }

    if state_key not in normalized_state_keys:
        raise KeyError(
            f"State '{STATE_NAME}' not found in crop_occurrences.json. "
            f"Available keys include: {list(raw.keys())[:20]}"
        )

    state_payload = raw[normalized_state_keys[state_key]]
    if not isinstance(state_payload, dict):
        raise ValueError(
            f"State '{STATE_NAME}' in crop_occurrences.json must map to an object."
        )

    result: Dict[str, Dict[str, Any]] = {}

    for crop_name, entry in state_payload.items():
        crop = normalize_name(crop_name)
        if not crop:
            continue

        if not isinstance(entry, dict):
            entry = {}

        result[crop] = {
            "occurrence": int(entry.get("occurrence", 0) or 0),
            "disease": dict(entry.get("disease", {}) or {}),
            "pests": dict(entry.get("pests", {}) or {}),
            "management": dict(entry.get("management", {}) or {}),
        }

        if entry.get("added"):
            result[crop]["added"] = True

    return result

CROP_OCCURRENCE_STATE = load_crop_occurrence_state(
    GLOBAL_CROP_OCCURRENCE_JSON,
    STATE_KEY,
)

CROP_OCCURRENCE_MAP = {
    crop: int(entry.get("occurrence", 0) or 0)
    for crop, entry in CROP_OCCURRENCE_STATE.items()
}
CROP_LIST = sorted(CROP_OCCURRENCE_MAP)

def initialize_crop_dictionary() -> Dict[str, Any]:
    return {
        STATE_KEY: {
            crop: {
                "disease": dict(entry.get("disease", {}) or {}),
                "pests": dict(entry.get("pests", {}) or {}),
                "management": dict(entry.get("management", {}) or {}),
                "occurrence": int(entry.get("occurrence", 0) or 0),
                **({"added": True} if entry.get("added") else {}),
            }
            for crop, entry in CROP_OCCURRENCE_STATE.items()
        }
    }

def build_crop_dictionary_from_ledger() -> Dict[str, Any]:
    crop_dict = initialize_crop_dictionary()

    with get_db() as conn:
        rows = conn.execute(
            """
            SELECT qualification_tag, qualification_entities_json
            FROM run_documents
            WHERE run_id=?
              AND duplicate=0
              AND qualification_status='succeeded'
              AND qualification_tag != 'msc'
            """,
            (RUN_ID,),
        ).fetchall()

    for row in rows:
        entities = json.loads(row["qualification_entities_json"] or "{}")

        for crop, fields in entities.items():
            crop = normalize_name(crop)
            if not crop:
                continue

            if crop not in crop_dict[STATE_KEY]:
                crop_dict[STATE_KEY][crop] = {
                    "disease": {},
                    "pests": {},
                    "management": {},
                    "occurrence": CROP_OCCURRENCE_MAP.get(crop, 0),
                    "added": True,
                }

            for field in ENTITY_FIELDS:
                unique_entities = {
                    ent
                    for ent in (
                        normalize_entity(x)
                        for x in fields.get(field, []) or []
                    )
                    if ent
                }

                for ent in unique_entities:
                    current = crop_dict[STATE_KEY][crop][field].get(ent, 0)
                    crop_dict[STATE_KEY][crop][field][ent] = current + 1

    for crop, entry in crop_dict[STATE_KEY].items():
        for field in ENTITY_FIELDS:
            entry[field] = dict(sorted(entry[field].items()))

    return crop_dict

def write_state_crop_occurrences(
    crop_dictionary: Dict[str, Any],
) -> Path:
    """Write only this state's crop artifact. Never mutate the global crop JSON here."""
    state_payload = {
        "schema_version": "1.0",
        "build_id": BUILD_ID,
        "wave_id": WAVE_ID,
        "state_name": STATE_NAME,
        "state_code": STATE_CODE,
        "state_key": STATE_KEY,
        "generated_at": utc_now(),
        "crop_occurrences": crop_dictionary.get(STATE_KEY, {}),
    }
    write_json_atomic(STATE_CROP_JSON, state_payload)
    return STATE_CROP_JSON

print(f"Loaded {len(CROP_LIST)} crop names for qualification.")


## 8. Source extraction + canonicalization

Successful extraction returns one normalized canonical document:

- PDF → one canonical document, while preserving page boundaries in metadata.
- Web URL → one canonical document.
- CSV row → one canonical document.

`month_year` is populated only when a credible source date is determinable; otherwise it is `""`.

In [ ]:
_thread_local = threading.local()

JUNK_URL_PATTERNS = [
    "youtube.com",
    "youtu.be",
    "youtube-nocookie.com",
    "facebook.com",
    "instagram.com",
    "linkedin.com",
    "twitter.com",
    "x.com/",
]

def get_http_session() -> requests.Session:
    if not hasattr(_thread_local, "session"):
        session = requests.Session()
        retries = Retry(
            total=3,
            connect=3,
            read=3,
            status=3,
            backoff_factor=1,
            status_forcelist=[429, 500, 502, 503, 504],
            allowed_methods=["GET", "HEAD"],
            raise_on_status=False,
        )
        adapter = HTTPAdapter(
            pool_connections=100,
            pool_maxsize=100,
            max_retries=retries,
            pool_block=False,
        )
        session.mount("http://", adapter)
        session.mount("https://", adapter)
        _thread_local.session = session
    return _thread_local.session

def is_junk_url(url: str) -> bool:
    u = url.lower()
    return any(pattern in u for pattern in JUNK_URL_PATTERNS)

def clean_text_common(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\x00", " ")
    text = re.sub(r"\r", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    ref_match = re.search(
        r"\n\s*(references|bibliography|literature cited)\s*\n",
        text,
        flags=re.IGNORECASE,
    )
    if ref_match and ref_match.start() > len(text) * 0.55:
        text = text[:ref_match.start()]
    return text.strip()

def normalize_month_year(value: Any) -> str:
    raw = str(value or "").strip()
    if not raw:
        return ""

    m = re.search(r"\b(19|20)\d{2}[-/](0[1-9]|1[0-2])\b", raw)
    if m:
        return m.group(0).replace("/", "-")[:7]

    # PDF dates often look like D:20240115103000.
    m = re.search(r"(?:D:)?((?:19|20)\d{2})(0[1-9]|1[0-2])", raw)
    if m:
        return f"{m.group(1)}-{m.group(2)}"

    try:
        dt = date_parser.parse(raw, fuzzy=False)
        if 1900 <= dt.year <= datetime.now().year + 1:
            return f"{dt.year:04d}-{dt.month:02d}"
    except Exception:
        pass
    return ""

def extract_web_month_year(soup: BeautifulSoup) -> str:
    candidates: List[str] = []

    meta_keys = [
        ("property", "article:published_time"),
        ("property", "article:modified_time"),
        ("name", "date"),
        ("name", "pubdate"),
        ("name", "publishdate"),
        ("name", "publication_date"),
        ("name", "dc.date"),
        ("itemprop", "datePublished"),
        ("itemprop", "dateModified"),
    ]
    for attr, key in meta_keys:
        tag = soup.find("meta", attrs={attr: key})
        if tag and tag.get("content"):
            candidates.append(tag.get("content"))

    for tag in soup.find_all("time"):
        if tag.get("datetime"):
            candidates.append(tag.get("datetime"))

    # JSON-LD is often the cleanest publication date.
    for script in soup.find_all("script", attrs={"type": "application/ld+json"}):
        try:
            payload = json.loads(script.string or "")
        except Exception:
            continue

        stack = payload if isinstance(payload, list) else [payload]
        for obj in stack:
            if isinstance(obj, dict):
                for key in ["datePublished", "dateCreated", "dateModified"]:
                    if obj.get(key):
                        candidates.append(str(obj[key]))

    for candidate in candidates:
        normalized = normalize_month_year(candidate)
        if normalized:
            return normalized
    return ""

def fallback_title_from_url(url: str) -> str:
    parsed = urlparse(url)
    slug = unquote(parsed.path.rstrip("/").split("/")[-1] or parsed.netloc)
    slug = re.sub(r"[-_]+", " ", slug).strip()
    return slug or parsed.netloc or "Untitled web page"

def extract_url_document(url: str) -> Dict[str, Any]:
    if is_junk_url(url):
        raise ValueError("junk_url_prefilter")

    response = get_http_session().get(
        url,
        timeout=(8, 30),
        headers={"User-Agent": "Mozilla/5.0 (MetaMIRAGE offline research preload)"},
        allow_redirects=True,
    )
    response.raise_for_status()

    content_type = response.headers.get("Content-Type", "").lower()
    if "text/html" not in content_type:
        raise ValueError(f"non_html_content_type:{content_type}")

    html = response.text
    if not html or len(html) < 200:
        raise ValueError("empty_or_too_short_html")

    soup = BeautifulSoup(html, "lxml")

    title = ""
    if soup.title and soup.title.get_text(strip=True):
        title = soup.title.get_text(" ", strip=True)
    if not title:
        h1 = soup.find("h1")
        if h1:
            title = h1.get_text(" ", strip=True)
    title = re.sub(r"\s+", " ", title).strip() or fallback_title_from_url(url)

    month_year = extract_web_month_year(soup) or normalize_month_year(response.headers.get("Last-Modified"))

    try:
        readable_html = Document(html).summary()
    except Exception:
        readable_html = html

    extracted = trafilatura.extract(
        readable_html,
        include_comments=False,
        include_tables=False,
        include_links=False,
        include_images=False,
        favor_precision=True,
        deduplicate=True,
    )
    if not extracted:
        extracted = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=False,
            include_links=False,
            include_images=False,
        )

    text = clean_text_common(extracted or "")
    if len(text) < 100:
        raise ValueError("web_extraction_empty_or_too_short")

    return {
        "text": text,
        "title": title,
        "month_year": month_year,
        "url": response.url or url,
        "raw_hash": stable_hash(html),
        "pages": [{"page": -1, "start": 0, "end": len(text)}],
        "source_metadata": {
            "requested_url": url,
            "final_url": response.url,
            "http_last_modified": response.headers.get("Last-Modified", ""),
        },
    }

def extract_pdf_document(path: Path) -> Dict[str, Any]:
    raw_hash = bytes_sha256(path)
    page_records = []
    joined_parts: List[str] = []
    cursor = 0

    with pdfplumber.open(str(path)) as pdf:
        metadata = pdf.metadata or {}
        title = re.sub(r"\s+", " ", str(metadata.get("Title") or "")).strip()
        title = title or path.stem
        month_year = (
            normalize_month_year(metadata.get("CreationDate"))
            or normalize_month_year(metadata.get("ModDate"))
        )

        for page_index, page in enumerate(pdf.pages):
            text = clean_text_common(page.extract_text() or "")
            if not text:
                continue

            if joined_parts:
                cursor += 2  # "\n\n"
            start = cursor
            joined_parts.append(text)
            cursor += len(text)
            end = cursor

            page_records.append({
                "page": page_index,
                "start": start,
                "end": end,
            })

    full_text = "\n\n".join(joined_parts).strip()
    if len(full_text) < 100:
        raise ValueError("pdf_extraction_empty_or_too_short")

    return {
        "text": full_text,
        "title": title,
        "month_year": month_year,
        "url": "",
        "raw_hash": raw_hash,
        "pages": page_records,
        "source_metadata": {
            "original_filename": path.name,
            "page_count_with_text": len(page_records),
        },
    }

COMMON_DATE_FIELDS = [
    "month_year", "publication_date", "published_date",
    "publish_date", "date", "created_at", "updated_at",
]
COMMON_URL_FIELDS = ["url", "source_url", "link"]
COMMON_COUNTY_FIELDS = ["county", "county_name"]
COMMON_TITLE_FIELDS = ["title", "name", "crop_name", "entity_name"]

def first_nonempty(row: Dict[str, Any], fields: Sequence[str]) -> str:
    for field in fields:
        value = row.get(field)
        if value is not None and str(value).strip():
            return str(value).strip()
    return ""

def deterministic_csv_narrative(
    row: Dict[str, Any],
    include_fields: Optional[Sequence[str]] = None,
) -> str:
    if include_fields:
        keys = [k for k in include_fields if k in row]
    else:
        keys = sorted(row.keys(), key=lambda x: str(x).lower())

    lines = []
    for key in keys:
        value = row.get(key)
        if value is None or not str(value).strip():
            continue
        clean_key = re.sub(r"\s+", " ", str(key).strip())
        clean_value = re.sub(r"\s+", " ", str(value).strip())
        lines.append(f"{clean_key}: {clean_value}")

    return clean_text_common("\n".join(lines))

def extract_csv_row_document(
    row: Dict[str, Any],
    row_number: int,
    csv_path: Path,
    config: Dict[str, Any],
) -> Dict[str, Any]:
    text = deterministic_csv_narrative(row, config.get("include_fields"))
    if not text:
        raise ValueError("empty_csv_row")

    title_fields = config.get("title_fields") or COMMON_TITLE_FIELDS
    title_parts = [
        str(row.get(f)).strip()
        for f in title_fields
        if row.get(f) is not None and str(row.get(f)).strip()
    ]
    title = " - ".join(title_parts[:3])
    if not title:
        title = f"{STATE_NAME} {csv_path.stem} record {row_number}"

    date_field = config.get("date_field")
    raw_date = row.get(date_field) if date_field else first_nonempty(row, COMMON_DATE_FIELDS)
    month_year = normalize_month_year(raw_date)

    url_field = config.get("url_field")
    url = (
        str(row.get(url_field) or "").strip()
        if url_field
        else first_nonempty(row, COMMON_URL_FIELDS)
    )

    county_field = config.get("county_field")
    county = (
        str(row.get(county_field) or "").strip()
        if county_field
        else first_nonempty(row, COMMON_COUNTY_FIELDS)
    )

    canonical_raw = json.dumps(
        {str(k): "" if v is None else str(v) for k, v in row.items()},
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    return {
        "text": text,
        "title": title,
        "month_year": month_year,
        "url": url,
        "raw_hash": stable_hash(canonical_raw),
        "pages": [{"page": -1, "start": 0, "end": len(text)}],
        "source_metadata": {
            "csv_file": str(csv_path),
            "csv_row_index": row_number,
            "county": county,
        },
    }

print("Extraction helpers ready.")

## 9. Source discovery, coordinator-backed global document dedupe, and state-local canonical persistence

Extraction remains concurrent inside one worker. After canonical text is produced, the worker atomically claims the document `content_hash` through the coordinator before committing ownership locally. This replaces the old shared-ledger cross-state dedupe path.


In [ ]:
def source_key_for(source_type: str, source_uri: str) -> str:
    return stable_hash(f"{RUN_ID}|{source_type}|{source_uri}")

def register_source_task(
    source_type: str,
    source_uri: str,
    payload: Optional[Dict[str, Any]] = None,
):
    key = source_key_for(source_type, source_uri)
    now = utc_now()
    with get_db() as conn:
        conn.execute(
            """
            INSERT OR IGNORE INTO source_tasks(
                source_key, run_id, source_type, source_uri,
                source_payload_json, status, created_at, updated_at
            ) VALUES (?, ?, ?, ?, ?, 'discovered', ?, ?)
            """,
            (
                key, RUN_ID, source_type, source_uri,
                json.dumps(payload, ensure_ascii=False) if payload is not None else None,
                now, now,
            ),
        )
    return key

def _dedupe_urls(urls: Iterable[str]) -> List[str]:
    result = []
    seen = set()

    for url in urls:
        url = str(url or "").strip().rstrip(",;")
        if url and url not in seen:
            seen.add(url)
            result.append(url)

    return result

def _extract_urls_from_values(values: Iterable[Any]) -> List[str]:
    urls: List[str] = []

    for value in values:
        text = str(value or "").strip()
        if not text:
            continue

        urls.extend(re.findall(r"https?://[^\s<>\"']+", text))

    return _dedupe_urls(urls)

def read_url_file(path: Path) -> List[str]:
    suffix = path.suffix.lower()

    if suffix == ".txt":
        values: List[str] = []
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith("#"):
                    continue
                values.append(line)

        return _extract_urls_from_values(values)

    if suffix in {".xlsx", ".xlsm"}:
        workbook = openpyxl.load_workbook(
            path,
            read_only=False,
            data_only=False,
        )
        values: List[Any] = []

        try:
            for worksheet in workbook.worksheets:
                for row in worksheet.iter_rows():
                    for cell in row:
                        if cell.value is not None:
                            values.append(cell.value)

                        if cell.hyperlink and cell.hyperlink.target:
                            values.append(cell.hyperlink.target)
        finally:
            workbook.close()

        return _extract_urls_from_values(values)

    if suffix == ".xls":
        workbook = xlrd.open_workbook(str(path), on_demand=True)
        values: List[Any] = []

        try:
            for worksheet in workbook.sheets():
                for row_index in range(worksheet.nrows):
                    values.extend(worksheet.row_values(row_index))
        finally:
            workbook.release_resources()

        return _extract_urls_from_values(values)

    raise ValueError(
        f"Unsupported URL file type: {path.suffix}. "
        "Expected .txt, .xlsx, .xlsm, or .xls."
    )

def discover_sources():
    discovered = 0

    if URL_FILE:
        for url in read_url_file(URL_FILE):
            register_source_task("web", url)
            discovered += 1
            if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                break

    if not (DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT):
        pdf_roots: List[Path] = []

        if PDF_ZIP_FILE:
            extracted_dir = INPUT_STAGING_DIR / "pdf_zip_extracted"
            extracted_dir.mkdir(parents=True, exist_ok=True)
            marker = extracted_dir / ".extracted_complete"
            if not marker.exists():
                with zipfile.ZipFile(PDF_ZIP_FILE, "r") as zf:
                    zf.extractall(extracted_dir)
                marker.write_text(utc_now(), encoding="utf-8")
            pdf_roots.append(extracted_dir)

        if PDF_DIR:
            pdf_roots.append(PDF_DIR)

        seen_pdf_paths = set()
        for root in pdf_roots:
            for path in sorted(
                p for p in root.rglob("*")
                if p.is_file() and p.suffix.lower() == ".pdf"
            ):
                resolved = str(path.resolve())
                if resolved in seen_pdf_paths:
                    continue
                seen_pdf_paths.add(resolved)
                register_source_task("pdf", resolved)
                discovered += 1
                if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                    break
            if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                break

    if not (DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT):
        for cfg in CSV_INPUTS:
            csv_path = Path(cfg["path"]).resolve()
            with open(csv_path, newline="", encoding="utf-8", errors="ignore") as f:
                reader = csv.DictReader(f)
                for row_number, row in enumerate(reader, start=1):
                    source_uri = f"{csv_path}#row={row_number}"
                    register_source_task(
                        "csv",
                        source_uri,
                        payload={
                            "row": row,
                            "row_number": row_number,
                            "csv_path": str(csv_path),
                            "config": {
                                k: v
                                for k, v in cfg.items()
                                if k != "path"
                            },
                        },
                    )
                    discovered += 1
                    if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                        break
            if DEBUG_SOURCE_LIMIT and discovered >= DEBUG_SOURCE_LIMIT:
                break

    with get_db() as conn:
        total = conn.execute(
            "SELECT COUNT(*) FROM source_tasks WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()[0]
    print(f"Source discovery complete. Registered for this run: {total}")

def _find_duplicate_by_raw_hash(raw_hash: str) -> Optional[str]:
    """State-local fast-path only. Cross-state authority is the coordinator content hash claim."""
    if not raw_hash:
        return None
    with get_db() as conn:
        row = conn.execute(
            "SELECT document_id FROM documents WHERE raw_hash=? LIMIT 1",
            (raw_hash,),
        ).fetchone()
    return row["document_id"] if row else None

def _register_duplicate(
    source_row: sqlite3.Row,
    existing_document_id: str,
    *,
    local_document: bool,
):
    """Mark the source duplicate. Only create run_documents when the document exists locally."""
    now = utc_now()
    with get_db() as conn:
        if local_document:
            conn.execute(
                """
                INSERT OR IGNORE INTO run_documents(
                    run_id, document_id, source_type, source_uri,
                    duplicate, document_status, created_at, updated_at
                ) VALUES (?, ?, ?, ?, 1, 'duplicate_skipped', ?, ?)
                """,
                (
                    RUN_ID, existing_document_id, source_row["source_type"],
                    source_row["source_uri"], now, now,
                ),
            )
        conn.execute(
            """
            UPDATE source_tasks
            SET status='duplicate_skipped', document_id=?, duplicate=1,
                source_payload_json=NULL, last_error=NULL, updated_at=?
            WHERE source_key=?
            """,
            (existing_document_id, now, source_row["source_key"]),
        )

def _restore_owner_source_status(source_row: sqlite3.Row, document_id: str):
    """Recover after a crash that committed the local canonical row before source completion."""
    with get_db() as conn:
        conn.execute(
            """
            UPDATE source_tasks
            SET status='extracted', document_id=?, duplicate=0,
                source_payload_json=NULL, last_error=NULL, updated_at=?
            WHERE source_key=?
            """,
            (document_id, utc_now(), source_row["source_key"]),
        )

def _persist_new_canonical_document(
    source_row: sqlite3.Row,
    extraction: Dict[str, Any],
) -> str:
    text = extraction["text"]
    doc_hash = content_hash(text)
    document_id = doc_hash

    # Threads in one notebook can still discover equal content concurrently. Serialize
    # the local canonical commit while the coordinator provides cross-worker atomicity.
    with CANONICAL_PERSIST_LOCK:
        with get_db() as conn:
            duplicate = conn.execute(
                "SELECT document_id FROM documents WHERE content_hash=?",
                (doc_hash,),
            ).fetchone()

        if duplicate:
            existing_document_id = duplicate["document_id"]
            if source_row["document_id"] == existing_document_id:
                # Recovery path: this source had already committed local ownership before
                # a later coordinator/source-status operation failed.
                claim = coordinator_claim_content(
                    "document", doc_hash, resource_id=existing_document_id
                )
                claim_status = _normalized_status(claim)
                owner_state = str(claim.get("owner_state") or "").upper()
                if claim_status in {"claimed", "already_owned"}:
                    coordinator_complete_content(
                        "document", doc_hash, resource_id=existing_document_id
                    )
                elif claim_status == "already_complete":
                    if owner_state and owner_state != STATE_CODE:
                        raise RuntimeError(
                            f"Local canonical owner conflict for {doc_hash}: coordinator owner={owner_state}"
                        )
                elif claim_status == "claimed_by_other":
                    raise RuntimeError(
                        f"Local canonical document {doc_hash} exists but coordinator claim is owned elsewhere: {claim}"
                    )
                _restore_owner_source_status(source_row, existing_document_id)
                return existing_document_id

            _register_duplicate(
                source_row, existing_document_id, local_document=True
            )
            return existing_document_id

        claim = coordinator_claim_content(
            "document", doc_hash, resource_id=document_id
        )
        claim_status = _normalized_status(claim)
        owner_state = str(claim.get("owner_state") or "").upper()
        claimed_resource = str(claim.get("resource_id") or "")

        if claim_status in {"already_complete", "claimed_by_other"}:
            # If the coordinator says this same state already completed the claim, allow
            # recovery to rebuild its local canonical state. A same-state claim owned by
            # another worker must never be treated as a duplicate; the coordinator should
            # transfer/recover stale claims when the current worker owns the state lease.
            same_state_recovery = (
                claim_status == "already_complete"
                and owner_state == STATE_CODE
                and (not claimed_resource or claimed_resource == document_id)
            )
            if claim_status == "claimed_by_other" and owner_state == STATE_CODE:
                raise RuntimeError(
                    "Coordinator returned a same-state document claim owned by another worker. "
                    "The current state-lease owner must recover/transfer stale content claims: "
                    f"{claim}"
                )
            if not same_state_recovery:
                _register_duplicate(
                    source_row, document_id, local_document=False
                )
                return document_id

        if claim_status not in {"claimed", "already_owned", "already_complete"}:
            raise RuntimeError(f"Unexpected document claim response: {claim}")

        text_path, metadata_path = canonical_paths(document_id)
        write_zstd_text(text_path, text)

        county = (extraction.get("source_metadata") or {}).get("county", "")
        location = STATE_NAME if not county else f"{STATE_NAME}, {county}"

        metadata = {
            "schema_version": "1.0",
            "document_id": document_id,
            "content_hash": doc_hash,
            "raw_hash": extraction.get("raw_hash", ""),
            "source_type": source_row["source_type"],
            "source_uri": source_row["source_uri"],
            "source_name": Path(source_row["source_uri"].split("#", 1)[0]).name
                if source_row["source_type"] in {"pdf", "csv"} else source_row["source_uri"],
            "run_discovered": RUN_ID,
            "build_id": BUILD_ID,
            "wave_id": WAVE_ID,
            "state_discovered": STATE_NAME,
            "state_code": STATE_CODE,
            "title": extraction["title"],
            "language": "en",
            "location": location,
            "month_year": extraction.get("month_year", ""),
            "url": extraction.get("url", ""),
            "canonical_text_path": str(text_path),
            "canonical_text_chars": len(text),
            "canonical_text_bytes": len(text.encode("utf-8")),
            "pages": extraction.get("pages", [{"page": -1, "start": 0, "end": len(text)}]),
            "extraction": {
                "extractor_version": EXTRACTOR_VERSION,
                "extracted_at": utc_now(),
            },
            "source_metadata": extraction.get("source_metadata", {}),
        }
        write_json_atomic(metadata_path, metadata)

        now = utc_now()
        with get_db() as conn:
            conn.execute(
                """
                INSERT OR IGNORE INTO documents(
                    document_id, content_hash, raw_hash, source_type,
                    canonical_text_path, canonical_metadata_path,
                    canonical_text_chars, canonical_text_bytes,
                    language, extractor_version, extraction_status,
                    first_seen_run_id, first_seen_state, created_at, updated_at
                ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'extracted', ?, ?, ?, ?)
                """,
                (
                    document_id, doc_hash, extraction.get("raw_hash", ""),
                    source_row["source_type"], str(text_path), str(metadata_path),
                    len(text), len(text.encode("utf-8")), "en",
                    EXTRACTOR_VERSION, RUN_ID, STATE_NAME, now, now,
                ),
            )
            conn.execute(
                """
                INSERT OR IGNORE INTO run_documents(
                    run_id, document_id, source_type, source_uri,
                    duplicate, document_status, created_at, updated_at
                ) VALUES (?, ?, ?, ?, 0, 'extracted', ?, ?)
                """,
                (
                    RUN_ID, document_id, source_row["source_type"],
                    source_row["source_uri"], now, now,
                ),
            )
            conn.execute(
                """
                UPDATE source_tasks
                SET status='extracted', document_id=?, duplicate=0,
                    source_payload_json=NULL, last_error=NULL, updated_at=?
                WHERE source_key=?
                """,
                (document_id, now, source_row["source_key"]),
            )

        # Content-claim completion means canonical ownership has been durably committed.
        # Qualification/indexing completion is tracked separately by the state lease/status.
        coordinator_complete_content(
            "document", doc_hash, resource_id=document_id
        )

    return document_id

def extract_source_task(source_row: sqlite3.Row) -> str:
    source_type = source_row["source_type"]
    source_uri = source_row["source_uri"]
    attempt_number = int(source_row["attempt_count"]) + 1
    started = utc_now()

    try:
        # Fast raw-file dedupe for PDFs, including renamed/moved copies.
        if source_type == "pdf":
            path = Path(source_uri)
            raw_hash = bytes_sha256(path)
            existing = _find_duplicate_by_raw_hash(raw_hash)
            if existing:
                _register_duplicate(source_row, existing, local_document=True)
                record_attempt("source", source_row["source_key"], "extract", attempt_number, "duplicate", started)
                return "duplicate_skipped"
            extraction = extract_pdf_document(path)

        elif source_type == "web":
            extraction = extract_url_document(source_uri)

        elif source_type == "csv":
            payload = json.loads(source_row["source_payload_json"] or "{}")
            row = payload["row"]
            csv_path = Path(payload["csv_path"])
            row_number = int(payload["row_number"])
            cfg = payload.get("config") or {}

            # Fast raw-record dedupe.
            canonical_raw = json.dumps(
                {str(k): "" if v is None else str(v) for k, v in row.items()},
                sort_keys=True,
                ensure_ascii=False,
                separators=(",", ":"),
            )
            raw_hash = stable_hash(canonical_raw)
            existing = _find_duplicate_by_raw_hash(raw_hash)
            if existing:
                _register_duplicate(source_row, existing, local_document=True)
                record_attempt("source", source_row["source_key"], "extract", attempt_number, "duplicate", started)
                return "duplicate_skipped"

            extraction = extract_csv_row_document(row, row_number, csv_path, cfg)

        else:
            raise ValueError(f"Unsupported source_type: {source_type}")

        _persist_new_canonical_document(source_row, extraction)
        record_attempt("source", source_row["source_key"], "extract", attempt_number, "succeeded", started)
        return "succeeded"

    except Exception as exc:
        with get_db() as conn:
            conn.execute(
                """
                UPDATE source_tasks
                SET status='failed', attempt_count=attempt_count+1,
                    last_error=?, updated_at=?
                WHERE source_key=?
                """,
                (traceback.format_exc()[-4000:], utc_now(), source_row["source_key"]),
            )
        record_attempt("source", source_row["source_key"], "extract", attempt_number, "failed", started, exc)
        return "failed"

def extraction_pass(statuses: Sequence[str] = ("discovered", "failed")):
    placeholders = ",".join("?" for _ in statuses)
    with get_db() as conn:
        rows = conn.execute(
            f"""
            SELECT * FROM source_tasks
            WHERE run_id=? AND status IN ({placeholders})
            ORDER BY created_at, source_key
            """,
            (RUN_ID, *statuses),
        ).fetchall()

    if not rows:
        print("No source tasks for this extraction pass.")
        return

    # URL/PDF extraction benefits from threads. CSV row conversion is cheap but safe in the same pool.
    with ThreadPoolExecutor(max_workers=EXTRACTION_WORKERS) as executor:
        futures = {executor.submit(extract_source_task, row): row["source_key"] for row in rows}
        counts = Counter()
        for future in tqdm(as_completed(futures), total=len(futures), desc="Extract + canonicalize"):
            try:
                counts[future.result()] += 1
            except Exception:
                counts["executor_error"] += 1
    print("Extraction pass:", dict(counts))

def retry_failed_extraction():
    for retry_index in range(MAX_STAGE_RETRIES):
        with get_db() as conn:
            failed = conn.execute(
                """
                SELECT COUNT(*) FROM source_tasks
                WHERE run_id=? AND status='failed'
                """,
                (RUN_ID,),
            ).fetchone()[0]
        if failed == 0:
            break
        print(f"Extraction retry {retry_index + 1}/{MAX_STAGE_RETRIES}: {failed} sources")
        extraction_pass(("failed",))

    with get_db() as conn:
        conn.execute(
            """
            UPDATE source_tasks
            SET status='permanently_failed', updated_at=?
            WHERE run_id=? AND status='failed'
            """,
            (utc_now(), RUN_ID),
        )

print("Discovery/extraction orchestration ready.")

## 10. Qualification model + original classifier contract

In [ ]:
classifier_tokenizer = None
classifier_model = None

def load_classifier_model():
    global classifier_tokenizer, classifier_model, HF_TOKEN

    if classifier_model is not None:
        return

    if not HF_TOKEN:
        HF_TOKEN = input("Paste your Hugging Face token: ").strip()

    if HF_TOKEN:
        login(token=HF_TOKEN)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    classifier_tokenizer = AutoTokenizer.from_pretrained(
        CLASSIFIER_MODEL_ID,
        token=HF_TOKEN or None,
    )
    classifier_model = AutoModelForCausalLM.from_pretrained(
        CLASSIFIER_MODEL_ID,
        token=HF_TOKEN or None,
        quantization_config=bnb_config,
        device_map="auto",
    )

    if classifier_tokenizer.pad_token is None:
        classifier_tokenizer.pad_token = classifier_tokenizer.eos_token

    print("Loaded classifier:", CLASSIFIER_MODEL_ID)

def unload_classifier_model():
    global classifier_tokenizer, classifier_model
    classifier_model = None
    classifier_tokenizer = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Classifier unloaded.")

In [ ]:
SYSTEM_PROMPT = """
You are an information extraction system.

Your task is to analyze agricultural content and return ONLY structured JSON output.

STRICT RULES:
- Output ONLY valid JSON. No explanations, no markdown, no extra text.
- Follow the schema exactly.
- Do NOT hallucinate information.
- Only extract information explicitly present in the text.
- If unsure, return empty lists.
- Do not include duplicate values.

CLASSIFICATION RULES:
You must classify the document chunk into ONE of:
- "crops"
- "pest"
- "disease"
- "management"
- "multi"
- "msc"

Definitions:
- "crops": general crop-related information such as growth, varieties, cultivation, planting, harvest, crop descriptions, or crop production.
- "pest": insects, mites, nematodes, animals, weeds, or other pests damaging crops.
- "disease": fungal, bacterial, viral, oomycete, nematode-caused, or physiological plant diseases/disorders.
- "management": treatment, prevention, farming practices, pesticide/herbicide/fungicide use, scouting, control strategies, irrigation, fertility, planting, harvest, storage, or integrated pest management.
- "multi": use only if multiple categories are clearly present.
- "msc": use if the content is not related to agriculture, crops, pests, diseases, or crop management.

TAG RULES:
- Use "multi" only if more than one category is strongly present.
- If "multi", provide the active categories in "subtags".
- If not "multi", "subtags" MUST be [].
- "subtags" may contain only: "crops", "pest", "disease", "management".
- Never put "multi" or "msc" inside subtags.

CROP RULES:
- You will be given a crop list.
- Match crops case-insensitively.
- Return crop names in lowercase.
- Return only crops that appear in the content.
- If a crop appears in the content but is not in the crop list, still include it.
- Do not invent crop names.

ENTITY EXTRACTION RULES:
- Extract only specific diseases, pests, and management practices explicitly present in the content.
- Normalize all extracted entities to lowercase.
- Keep entities short and specific.
- Do NOT include generic words such as "disease", "pest", "issue", "problem", "management", "control", "crop", or "plant".
- If no specific entity is present, return an empty list for that field.

RELATIONSHIP RULE:
- Preserve relationships between crops and entities.
- Each crop must only contain diseases, pests, and management practices relevant to that crop.
- Do NOT assign every entity to every crop unless the content clearly says the entity applies to all those crops.
- If the content explicitly discusses multiple crops separately, keep their entities separate.

STRICT CONSISTENCY:
- If tag = "disease", only fill "disease"; "pests" and "management" must be empty.
- If tag = "pest", only fill "pests"; "disease" and "management" must be empty.
- If tag = "management", only fill "management"; "disease" and "pests" must be empty.
- If tag = "crops", crop_entities may contain crops with empty entity lists.
- If tag = "multi", fill only fields corresponding to subtags.
- If tag = "msc", crop_entities must be {}.

OUTPUT SCHEMA:
{
  "tag": "string",
  "subtags": ["string"],
  "crop_entities": {
    "crop_name": {
      "disease": ["string"],
      "pests": ["string"],
      "management": ["string"]
    }
  }
}
"""

def build_user_prompt(crop_list: List[str], content: str) -> str:
    crop_text = json.dumps(crop_list, ensure_ascii=False)
    return f"""CROP LIST:
{crop_text}

CONTENT:
{content}
"""

In [ ]:
def build_user_prompt(crop_list: List[str], content: str) -> str:
    crop_text = json.dumps(crop_list, ensure_ascii=False)
    return f"""CROP LIST:
{crop_text}

CONTENT:
{content}
"""

def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not text:
        return None

    text = text.strip()
    text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"```$", "", text).strip()

    try:
        value = json.loads(text)
        return value if isinstance(value, dict) else None
    except Exception:
        pass

    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end > start:
        try:
            value = json.loads(text[start:end + 1])
            return value if isinstance(value, dict) else None
        except Exception:
            return None
    return None

def run_llm_once(content: str, crop_list: List[str], max_new_tokens: int = 900) -> str:
    if classifier_model is None:
        load_classifier_model()

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(crop_list, content)},
    ]
    prompt = classifier_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = classifier_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=12000,
    ).to(classifier_model.device)

    with torch.no_grad():
        output_ids = classifier_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=classifier_tokenizer.eos_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return classifier_tokenizer.decode(generated, skip_special_tokens=True).strip()

def fallback_msc() -> Dict[str, Any]:
    return {"tag": "msc", "subtags": [], "crop_entities": {}}

def crops_found_by_string_match(text: str, crop_list: List[str]) -> Set[str]:
    text_norm = normalize_name(text)
    found = set()
    for crop in crop_list:
        pattern = r"(?<![a-z0-9])" + re.escape(crop) + r"(?![a-z0-9])"
        if re.search(pattern, text_norm):
            found.add(crop)
    return found

def crop_appears_in_text(crop: str, text: str) -> bool:
    crop_norm = normalize_name(crop)
    text_norm = normalize_name(text)
    pattern = r"(?<![a-z0-9])" + re.escape(crop_norm) + r"(?![a-z0-9])"
    return re.search(pattern, text_norm) is not None

def validate_and_clean_llm_output(raw: Any, text: str, crop_list: List[str]) -> Dict[str, Any]:
    if not isinstance(raw, dict):
        raise ValueError("LLM output is not a JSON object")

    tag = normalize_name(raw.get("tag", ""))
    if tag not in ALLOWED_TAGS:
        raise ValueError(f"Invalid tag: {tag!r}")

    subtags = raw.get("subtags", [])
    if not isinstance(subtags, list):
        raise ValueError("subtags must be a list")
    subtags = sorted({
        normalize_name(s)
        for s in subtags
        if normalize_name(s) in {"crops", "pest", "disease", "management"}
    })

    if tag != "multi":
        subtags = []

    crop_entities = raw.get("crop_entities", {})
    if tag == "msc":
        return fallback_msc()
    if not isinstance(crop_entities, dict):
        raise ValueError("crop_entities must be an object")

    valid_known_crops_in_text = crops_found_by_string_match(text, crop_list)
    cleaned_crop_entities = {}

    for crop_name, fields in crop_entities.items():
        crop_norm = normalize_name(crop_name)
        if not crop_norm:
            continue

        if crop_norm in crop_list:
            if crop_norm not in valid_known_crops_in_text:
                continue
        elif not crop_appears_in_text(crop_norm, text):
            continue

        if not isinstance(fields, dict):
            fields = {}

        cleaned_fields = {"disease": [], "pests": [], "management": []}
        for field in ENTITY_FIELDS:
            values = fields.get(field, [])
            if not isinstance(values, list):
                values = []
            cleaned_fields[field] = sorted({
                ent
                for ent in (normalize_entity(v) for v in values)
                if ent
            })

        cleaned_crop_entities[crop_norm] = cleaned_fields

    if tag == "multi":
        allowed_fields = set()
        if "disease" in subtags:
            allowed_fields.add("disease")
        if "pest" in subtags:
            allowed_fields.add("pests")
        if "management" in subtags:
            allowed_fields.add("management")
    else:
        allowed_fields = {
            "crops": set(),
            "disease": {"disease"},
            "pest": {"pests"},
            "management": {"management"},
        }.get(tag, set())

    for fields in cleaned_crop_entities.values():
        for field in ENTITY_FIELDS:
            if field not in allowed_fields:
                fields[field] = []

    if tag == "multi":
        inferred = set(subtags)
        for fields in cleaned_crop_entities.values():
            if fields["disease"]:
                inferred.add("disease")
            if fields["pests"]:
                inferred.add("pest")
            if fields["management"]:
                inferred.add("management")
        subtags = sorted(inferred)

        if len(subtags) < 2:
            for candidate in ["disease", "pest", "management", "crops"]:
                if candidate in subtags:
                    tag, subtags = candidate, []
                    break
            else:
                return fallback_msc()

    return {
        "tag": tag,
        "subtags": subtags,
        "crop_entities": cleaned_crop_entities,
    }

def meaningful_categories_from_output(output: Dict[str, Any]) -> Set[str]:
    tag = output.get("tag", "msc")
    if tag == "multi":
        return set(output.get("subtags", [])) - {"msc", "multi"}
    if tag in {"crops", "pest", "disease", "management"}:
        return {tag}
    return set()

def merge_validated_outputs(outputs: Sequence[Dict[str, Any]]) -> Dict[str, Any]:
    categories = set()
    merged_entities = defaultdict(
        lambda: {"disease": set(), "pests": set(), "management": set()}
    )

    for validated in outputs:
        categories.update(meaningful_categories_from_output(validated))
        for crop, fields in (validated.get("crop_entities") or {}).items():
            crop_norm = normalize_name(crop)
            if not crop_norm:
                continue
            for field in ENTITY_FIELDS:
                for ent in fields.get(field, []) or []:
                    norm_ent = normalize_entity(ent)
                    if norm_ent:
                        merged_entities[crop_norm][field].add(norm_ent)

    if not categories:
        return fallback_msc()

    final_tag = next(iter(categories)) if len(categories) == 1 else "multi"
    subtags = [] if final_tag != "multi" else sorted(categories)

    crop_entities = {
        crop: {
            "disease": sorted(fields["disease"]),
            "pests": sorted(fields["pests"]),
            "management": sorted(fields["management"]),
        }
        for crop, fields in merged_entities.items()
    }

    allowed = set()
    if final_tag == "disease":
        allowed = {"disease"}
    elif final_tag == "pest":
        allowed = {"pests"}
    elif final_tag == "management":
        allowed = {"management"}
    elif final_tag == "multi":
        if "disease" in subtags:
            allowed.add("disease")
        if "pest" in subtags:
            allowed.add("pests")
        if "management" in subtags:
            allowed.add("management")

    for fields in crop_entities.values():
        for field in ENTITY_FIELDS:
            if field not in allowed:
                fields[field] = []

    return {
        "tag": final_tag,
        "subtags": subtags,
        "crop_entities": crop_entities,
    }

def total_entity_count(doc_output: Dict[str, Any]) -> int:
    return sum(
        len(fields.get(field, []))
        for fields in doc_output.get("crop_entities", {}).values()
        for field in ENTITY_FIELDS
    )

def has_crop_and_entity(doc_output: Dict[str, Any]) -> bool:
    return bool(doc_output.get("crop_entities")) and total_entity_count(doc_output) >= 1

print("Qualification contract ready.")

## 11. Qualification chunk persistence, cache, retries, and document decision

The initial pass processes all documents. Failed qualification chunks are retried **after** the pass, scoped to this `RUN_ID`. Remaining failures become `permanently_failed`.

A document can still be decided from successful chunks if some sibling chunks permanently fail; those failures remain auditable in the manifest.

In [ ]:
def qualification_chunks_for_text(text: str) -> List[str]:
    text = text.strip()
    if not text:
        return []

    chunks = []
    start = 0
    while start < len(text) and len(chunks) < MAX_QUALIFICATION_CHUNKS:
        end = min(start + QUALIFICATION_CHUNK_CHARS, len(text))
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - QUALIFICATION_OVERLAP_CHARS)
    return chunks

def ensure_qualification_chunk_rows(document_id: str):
    with get_db() as conn:
        doc = conn.execute(
            "SELECT canonical_text_path FROM documents WHERE document_id=?",
            (document_id,),
        ).fetchone()
    text = read_zstd_text(Path(doc["canonical_text_path"]))
    chunks = qualification_chunks_for_text(text)

    now = utc_now()
    with get_db() as conn:
        for i, chunk in enumerate(chunks):
            qid = stable_hash(f"{RUN_ID}|{document_id}|q|{i}|{CLASSIFIER_VERSION}")
            conn.execute(
                """
                INSERT OR IGNORE INTO qualification_chunks(
                    qualification_chunk_id, run_id, document_id,
                    chunk_index, chunk_hash, status,
                    created_at, updated_at
                ) VALUES (?, ?, ?, ?, ?, 'pending', ?, ?)
                """,
                (qid, RUN_ID, document_id, i, content_hash(chunk), now, now),
            )

def qualification_cache_key(chunk: str) -> Tuple[str, str, str]:
    crop_hash = stable_hash(json.dumps(CROP_LIST, sort_keys=True))
    chunk_content_hash = content_hash(chunk)
    payload = {
        "model_id": CLASSIFIER_MODEL_ID,
        "classifier_version": CLASSIFIER_VERSION,
        "state": STATE_KEY,
        "crop_hash": crop_hash,
        "content_hash": chunk_content_hash,
    }
    return stable_hash(json.dumps(payload, sort_keys=True)), crop_hash, chunk_content_hash

def classify_chunk_one_attempt(chunk: str) -> Tuple[Dict[str, Any], str]:
    cache_key, crop_hash, chunk_content_hash = qualification_cache_key(chunk)

    with get_db() as conn:
        cached = conn.execute(
            "SELECT * FROM classification_cache WHERE cache_key=?",
            (cache_key,),
        ).fetchone()

    if cached:
        return json.loads(cached["validated_output_json"]), cached["raw_output"] or ""

    raw_text = run_llm_once(chunk, CROP_LIST)
    parsed = extract_json_object(raw_text)
    if parsed is None:
        raise ValueError("Classifier did not return parseable JSON")

    cleaned = validate_and_clean_llm_output(parsed, chunk, CROP_LIST)

    with get_db() as conn:
        conn.execute(
            """
            INSERT OR REPLACE INTO classification_cache(
                cache_key, model_id, classifier_version, state_key,
                crop_hash, content_hash, validated_output_json,
                raw_output, created_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                cache_key, CLASSIFIER_MODEL_ID, CLASSIFIER_VERSION,
                STATE_KEY, crop_hash, chunk_content_hash,
                json.dumps(cleaned, ensure_ascii=False), raw_text, utc_now(),
            ),
        )

    return cleaned, raw_text

def _load_document_text(document_id: str) -> str:
    with get_db() as conn:
        row = conn.execute(
            "SELECT canonical_text_path FROM documents WHERE document_id=?",
            (document_id,),
        ).fetchone()
    if row is None:
        raise KeyError(document_id)
    return read_zstd_text(Path(row["canonical_text_path"]))

def _successful_doc_outputs(document_id: str) -> List[Dict[str, Any]]:
    with get_db() as conn:
        rows = conn.execute(
            """
            SELECT classification_result_json
            FROM qualification_chunks
            WHERE run_id=? AND document_id=? AND status='succeeded'
            ORDER BY chunk_index
            """,
            (RUN_ID, document_id),
        ).fetchall()
    return [
        json.loads(row["classification_result_json"])
        for row in rows
        if row["classification_result_json"]
    ]

def process_qualification_chunk(row: sqlite3.Row) -> str:
    document_id = row["document_id"]
    text = _load_document_text(document_id)
    chunks = qualification_chunks_for_text(text)
    idx = int(row["chunk_index"])

    if idx >= len(chunks):
        raise IndexError(f"Qualification chunk index {idx} no longer exists")

    chunk = chunks[idx]
    attempt_number = int(row["attempt_count"]) + 1
    started = utc_now()

    try:
        cleaned, raw_text = classify_chunk_one_attempt(chunk)

        with get_db() as conn:
            conn.execute(
                """
                UPDATE qualification_chunks
                SET status='succeeded',
                    classification_result_json=?,
                    attempt_count=attempt_count+1,
                    last_error=NULL,
                    updated_at=?
                WHERE qualification_chunk_id=?
                """,
                (
                    json.dumps(cleaned, ensure_ascii=False),
                    utc_now(), row["qualification_chunk_id"],
                ),
            )
        record_attempt(
            "qualification_chunk", row["qualification_chunk_id"],
            "classify", attempt_number, "succeeded", started,
        )
        return "succeeded"

    except Exception as exc:
        with get_db() as conn:
            conn.execute(
                """
                UPDATE qualification_chunks
                SET status='failed',
                    attempt_count=attempt_count+1,
                    last_error=?,
                    updated_at=?
                WHERE qualification_chunk_id=?
                """,
                (
                    traceback.format_exc()[-4000:],
                    utc_now(), row["qualification_chunk_id"],
                ),
            )
        record_attempt(
            "qualification_chunk", row["qualification_chunk_id"],
            "classify", attempt_number, "failed", started, exc,
        )
        return "failed"

def maybe_apply_early_stop(document_id: str):
    outputs = _successful_doc_outputs(document_id)
    merged = merge_validated_outputs(outputs)

    if (
        total_entity_count(merged) >= EARLY_STOP_TOTAL_ENTITIES
        or has_crop_and_entity(merged)
    ):
        with get_db() as conn:
            conn.execute(
                """
                UPDATE qualification_chunks
                SET status='skipped_early_stop', updated_at=?
                WHERE run_id=? AND document_id=? AND status='pending'
                """,
                (utc_now(), RUN_ID, document_id),
            )
        return True
    return False

def qualification_initial_pass():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND duplicate=0
              AND document_status IN ('extracted', 'qualifying')
            ORDER BY created_at, document_id
            """,
            (RUN_ID,),
        ).fetchall()

    if not docs:
        print("No documents awaiting qualification.")
        return

    load_classifier_model()

    for doc_row in tqdm(docs, desc="Qualifying documents"):
        document_id = doc_row["document_id"]
        ensure_qualification_chunk_rows(document_id)

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status='qualifying',
                    qualification_status='processing',
                    classifier_version=?,
                    updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (CLASSIFIER_VERSION, utc_now(), RUN_ID, document_id),
            )
            chunks = conn.execute(
                """
                SELECT * FROM qualification_chunks
                WHERE run_id=? AND document_id=? AND status='pending'
                ORDER BY chunk_index
                """,
                (RUN_ID, document_id),
            ).fetchall()

        for row in chunks:
            process_qualification_chunk(row)
            maybe_apply_early_stop(document_id)

            with get_db() as conn:
                still_pending = conn.execute(
                    """
                    SELECT COUNT(*) FROM qualification_chunks
                    WHERE run_id=? AND document_id=? AND status='pending'
                    """,
                    (RUN_ID, document_id),
                ).fetchone()[0]
            if still_pending == 0:
                break

def retry_failed_qualification():
    for retry_idx in range(MAX_STAGE_RETRIES):
        with get_db() as conn:
            failed_rows = conn.execute(
                """
                SELECT * FROM qualification_chunks
                WHERE run_id=? AND status='failed'
                ORDER BY document_id, chunk_index
                """,
                (RUN_ID,),
            ).fetchall()

        if not failed_rows:
            break

        print(
            f"Qualification retry {retry_idx + 1}/{MAX_STAGE_RETRIES}: "
            f"{len(failed_rows)} failed chunks"
        )
        load_classifier_model()

        for row in tqdm(failed_rows, desc=f"Qualification retry {retry_idx + 1}"):
            process_qualification_chunk(row)

    with get_db() as conn:
        conn.execute(
            """
            UPDATE qualification_chunks
            SET status='permanently_failed', updated_at=?
            WHERE run_id=? AND status='failed'
            """,
            (utc_now(), RUN_ID),
        )

def finalize_qualification_decisions():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND duplicate=0
              AND document_status NOT IN (
                  'rejected', 'accepted', 'rag_preparing',
                  'indexing', 'indexed', 'permanently_failed'
              )
            """,
            (RUN_ID,),
        ).fetchall()

    for row in docs:
        document_id = row["document_id"]

        with get_db() as conn:
            nonterminal = conn.execute(
                """
                SELECT COUNT(*) FROM qualification_chunks
                WHERE run_id=? AND document_id=?
                  AND status IN ('pending','processing','failed')
                """,
                (RUN_ID, document_id),
            ).fetchone()[0]
            succeeded = conn.execute(
                """
                SELECT COUNT(*) FROM qualification_chunks
                WHERE run_id=? AND document_id=? AND status='succeeded'
                """,
                (RUN_ID, document_id),
            ).fetchone()[0]

        if nonterminal:
            continue

        if succeeded == 0:
            with get_db() as conn:
                conn.execute(
                    """
                    UPDATE run_documents
                    SET document_status='permanently_failed',
                        qualification_status='permanently_failed',
                        accepted=0,
                        qualification_reason='No qualification chunk succeeded',
                        updated_at=?
                    WHERE run_id=? AND document_id=?
                    """,
                    (utc_now(), RUN_ID, document_id),
                )
            continue

        merged = merge_validated_outputs(_successful_doc_outputs(document_id))
        accepted = merged["tag"] != "msc"

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status=?,
                    qualification_status='succeeded',
                    accepted=?,
                    qualification_tag=?,
                    qualification_subtags_json=?,
                    qualification_entities_json=?,
                    qualification_reason=?,
                    classifier_version=?,
                    updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (
                    "accepted" if accepted else "rejected",
                    int(accepted),
                    merged["tag"],
                    json.dumps(merged["subtags"], ensure_ascii=False),
                    json.dumps(merged["crop_entities"], ensure_ascii=False),
                    "tag != msc" if accepted else "tag == msc",
                    CLASSIFIER_VERSION,
                    utc_now(), RUN_ID, document_id,
                ),
            )

    print("Qualification decisions finalized.")

print("Qualification persistence/retry pipeline ready.")

## 12. Runtime-compatible RAG chunking

Contract:

- Tokenizer: `BAAI/bge-base-en-v1.5`
- Max RAG chunk: 480 tokens
- Overlap: 80
- Hard cap: 512

Source behavior:

- Web ≤512 tokens → one chunk; otherwise 480/80.
- PDF → each page chunked independently at 480/80.
- CSV row ≤512 → one chunk; otherwise 480/80.

In [ ]:
rag_tokenizer = None

def load_rag_tokenizer():
    global rag_tokenizer
    if rag_tokenizer is None:
        rag_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
    return rag_tokenizer

def token_chunks(text: str, max_tokens: int = RAG_CHUNK_SIZE, overlap: int = RAG_CHUNK_OVERLAP) -> List[str]:
    tokenizer = load_rag_tokenizer()
    max_tokens = min(int(max_tokens), RAG_HARD_CAP)
    tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end][:RAG_HARD_CAP]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True).strip()
        if chunk_text:
            chunks.append(chunk_text)

        if end == len(tokens):
            break
        start = max(end - overlap, 0)

    return chunks

def canonical_doc_metadata(document_id: str) -> Dict[str, Any]:
    with get_db() as conn:
        row = conn.execute(
            "SELECT canonical_metadata_path FROM documents WHERE document_id=?",
            (document_id,),
        ).fetchone()
    return load_json(Path(row["canonical_metadata_path"]), {})

def reconstruct_page_text(document_id: str, page: int) -> str:
    meta = canonical_doc_metadata(document_id)
    text = _load_document_text(document_id)

    if page == -1:
        return text

    for rec in meta.get("pages", []):
        if int(rec.get("page", -999)) == int(page):
            return text[int(rec["start"]):int(rec["end"])]
    raise KeyError(f"Page {page} not found in canonical metadata for {document_id}")

def build_rag_chunk_specs(document_id: str) -> List[Dict[str, Any]]:
    meta = canonical_doc_metadata(document_id)
    source_type = meta["source_type"]
    specs: List[Dict[str, Any]] = []

    if source_type == "pdf":
        for page_rec in meta.get("pages", []):
            page = int(page_rec["page"])
            page_text = reconstruct_page_text(document_id, page)
            chunks = token_chunks(page_text, RAG_CHUNK_SIZE, RAG_CHUNK_OVERLAP)
            for chunk_index, chunk in enumerate(chunks):
                specs.append({
                    "page": page,
                    "chunk_index": chunk_index,
                    "text": chunk,
                })
    elif source_type in {"web", "csv"}:
        text = reconstruct_page_text(document_id, -1)
        tokenizer = load_rag_tokenizer()
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False, truncation=False))
        chunks = [text] if n_tokens <= RAG_HARD_CAP else token_chunks(
            text, RAG_CHUNK_SIZE, RAG_CHUNK_OVERLAP
        )
        for chunk_index, chunk in enumerate(chunks):
            specs.append({
                "page": -1,
                "chunk_index": chunk_index,
                "text": chunk,
            })
    else:
        raise ValueError(f"Unsupported source_type for RAG chunking: {source_type}")

    return specs

print("RAG chunking helpers ready.")

## 13. Metadata enrichment + hard contract validation

This remains the ingestion gate. Every Qdrant point keeps the runtime metadata contract and additionally records build/wave/state provenance for concurrent preload auditing.


In [ ]:
REQUIRED_QDRANT_FIELDS = [
    "text",
    "chunk_id",
    "source_type",
    "source_id",
    "title",
    "url",
    "page",
    "chunk_index",
    "location",
    "month_year",
    "content_hash",
    "language",
    "hardiness_zone",
]

def build_chunk_id(
    source_id: str,
    page: int,
    chunk_index: int,
    chunk_content_hash: str,
) -> str:
    """Agreed logical ID: source_id + page + chunk_index + content_hash."""
    return f"{source_id}|p={int(page)}|c={int(chunk_index)}|h={chunk_content_hash}"

def enrich_chunk_metadata(
    document_id: str,
    page: int,
    chunk_index: int,
    chunk_text: str,
) -> Dict[str, Any]:
    meta = canonical_doc_metadata(document_id)

    title = re.sub(r"\s+", " ", str(meta.get("title") or "")).strip()
    if not title:
        if meta["source_type"] == "pdf":
            title = Path(meta["source_uri"]).stem
        elif meta["source_type"] == "web":
            title = fallback_title_from_url(meta.get("url") or meta["source_uri"])
        else:
            title = f"{STATE_NAME} record {document_id[:12]}"

    location = re.sub(r"\s+", " ", str(meta.get("location") or STATE_NAME)).strip()
    zone = hardiness_zone_for_location(location)

    source_id = document_id[:16]
    chunk_content_hash = content_hash(chunk_text)
    chunk_id = build_chunk_id(
        source_id,
        page,
        chunk_index,
        chunk_content_hash,
    )

    # Runtime ingestion stores a title-prefixed document while hashing the raw chunk.
    stored_text = f"Title: {title}\n\n{chunk_text}"

    return {
        "text": stored_text,
        "chunk_id": chunk_id,
        "source_type": meta["source_type"],
        "source_id": source_id,
        "title": title,
        "url": str(meta.get("url") or ""),
        "page": int(page),
        "chunk_index": int(chunk_index),
        "location": location,
        "month_year": normalize_month_year(meta.get("month_year")) or "",
        "content_hash": chunk_content_hash,
        "language": str(meta.get("language") or "en"),
        "hardiness_zone": zone,
        # Operational provenance; runtime retrieval does not depend on these fields.
        "build_id": BUILD_ID,
        "wave_id": WAVE_ID,
        "ingest_state": STATE_CODE,
    }

def validate_qdrant_metadata(payload: Dict[str, Any]) -> Dict[str, Any]:
    missing_fields = [field for field in REQUIRED_QDRANT_FIELDS if field not in payload]
    invalid_fields = []

    if not missing_fields:
        nonempty_required = [
            "text", "chunk_id", "source_type", "source_id",
            "title", "location", "content_hash", "language",
        ]
        for field in nonempty_required:
            if not isinstance(payload[field], str) or not payload[field].strip():
                invalid_fields.append(field)

        if not isinstance(payload["url"], str):
            invalid_fields.append("url")

        if not isinstance(payload["page"], int):
            invalid_fields.append("page")
        if not isinstance(payload["chunk_index"], int) or payload["chunk_index"] < 0:
            invalid_fields.append("chunk_index")

        month_year = payload["month_year"]
        if not isinstance(month_year, str):
            invalid_fields.append("month_year")
        elif month_year and not re.fullmatch(r"\d{4}-(0[1-9]|1[0-2])", month_year):
            invalid_fields.append("month_year")

        zone = payload["hardiness_zone"]
        if not isinstance(zone, str):
            invalid_fields.append("hardiness_zone")
        elif not zone.strip() and STATE_CODE not in {"AK", "HI"}:
            invalid_fields.append("hardiness_zone")

    return {
        "valid": not missing_fields and not invalid_fields,
        "missing_fields": sorted(set(missing_fields)),
        "invalid_fields": sorted(set(invalid_fields)),
        "month_year_available": bool(payload.get("month_year")),
        "ak_hi_hardiness_exception": (
            STATE_CODE in {"AK", "HI"} and not str(payload.get("hardiness_zone", "")).strip()
        ),
    }

def prepare_rag_chunks():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND accepted=1
              AND document_status IN ('accepted','rag_preparing','indexing')
            ORDER BY created_at, document_id
            """,
            (RUN_ID,),
        ).fetchall()

    for row in tqdm(docs, desc="Preparing RAG chunks"):
        document_id = row["document_id"]
        specs = build_rag_chunk_specs(document_id)

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status='rag_preparing', rag_status='preparing', updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (utc_now(), RUN_ID, document_id),
            )

        for spec in specs:
            page = int(spec["page"])
            chunk_index = int(spec["chunk_index"])
            chunk_text = spec["text"]
            chunk_hash = content_hash(chunk_text)

            payload = enrich_chunk_metadata(
                document_id, page, chunk_index, chunk_text
            )
            chunk_id = payload["chunk_id"]
            rag_chunk_id = stable_hash(f"{RUN_ID}|{CHUNKER_VERSION}|{chunk_id}")
            qdrant_point_id = deterministic_point_uuid(chunk_id)
            validation = validate_qdrant_metadata(payload)

            status = "metadata_validated" if validation["valid"] else "failed"
            failure_stage = None if validation["valid"] else "metadata"
            last_error = None if validation["valid"] else json.dumps(validation)

            token_count = len(
                load_rag_tokenizer().encode(
                    chunk_text,
                    add_special_tokens=False,
                    truncation=False,
                )
            )

            inserted = False
            metadata_started = utc_now()
            with get_db() as conn:
                cursor = conn.execute(
                    """
                    INSERT OR IGNORE INTO rag_chunks(
                        rag_chunk_id, run_id, document_id,
                        page, chunk_index, chunk_hash, token_count,
                        chunker_version, metadata_json,
                        status, embedding_status, qdrant_status,
                        metadata_attempt_count, failure_stage, last_error, qdrant_point_id,
                        created_at, updated_at
                    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 'pending', 'pending', 1, ?, ?, ?, ?, ?)
                    """,
                    (
                        rag_chunk_id, RUN_ID, document_id, page, chunk_index,
                        chunk_hash, token_count, CHUNKER_VERSION,
                        json.dumps(payload, ensure_ascii=False),
                        status, failure_stage, last_error, qdrant_point_id,
                        utc_now(), utc_now(),
                    ),
                )
                inserted = cursor.rowcount == 1

            if inserted:
                record_attempt(
                    "rag_chunk", rag_chunk_id, "metadata", 1,
                    "succeeded" if validation["valid"] else "failed",
                    metadata_started,
                    None if validation["valid"] else ValueError(json.dumps(validation)),
                )

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status='indexing', rag_status='prepared', updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (utc_now(), RUN_ID, document_id),
            )

    print("RAG chunk preparation complete.")

print("Metadata contract gate ready.")

## 14. Embedding + shared Qdrant validation

The sentence-transformer model matches runtime retrieval. The shared build collection must already exist on the service node. Workers may validate the schema/indexes but never reset, restore, delete, or snapshot the collection.


In [ ]:
embedding_model = None
qdrant_client = None

def load_embedding_model():
    global embedding_model
    if embedding_model is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
        embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
        print(
            "Loaded embedding model:",
            EMBEDDING_MODEL,
            "dimension=",
            embedding_model.get_sentence_embedding_dimension(),
            "device=",
            device,
        )
    return embedding_model

def get_qdrant_client() -> QdrantClient:
    global qdrant_client
    if qdrant_client is None:
        qdrant_client = QdrantClient(
            url=QDRANT_URL,
            api_key=QDRANT_API_KEY,
            timeout=120,
        )
    return qdrant_client

def qdrant_headers() -> Dict[str, str]:
    return {"api-key": QDRANT_API_KEY} if QDRANT_API_KEY else {}

def qdrant_collection_exists() -> bool:
    return get_qdrant_client().collection_exists(QDRANT_COLLECTION)

def qdrant_point_count() -> int:
    if not qdrant_collection_exists():
        return 0
    return int(
        get_qdrant_client().count(
            collection_name=QDRANT_COLLECTION,
            exact=True,
        ).count
    )

def _existing_collection_vector_size() -> Optional[int]:
    if not qdrant_collection_exists():
        return None
    info = get_qdrant_client().get_collection(QDRANT_COLLECTION)
    vectors = info.config.params.vectors
    if hasattr(vectors, "size"):
        return int(vectors.size)
    if isinstance(vectors, dict) and vectors:
        first = next(iter(vectors.values()))
        if hasattr(first, "size"):
            return int(first.size)
        if isinstance(first, dict) and "size" in first:
            return int(first["size"])
    return None

def ensure_payload_indexes():
    """Idempotently ensure runtime-required payload indexes on the shared collection."""
    client = get_qdrant_client()
    for field in ["hardiness_zone", "month_year", "title", "content_hash"]:
        try:
            client.create_payload_index(
                collection_name=QDRANT_COLLECTION,
                field_name=field,
                field_schema=PayloadSchemaType.KEYWORD,
                wait=True,
            )
        except Exception as exc:
            msg = str(exc).lower()
            if "already exists" not in msg and "already" not in msg:
                print(f"Payload index warning for {field}: {exc}")

def validate_shared_qdrant_collection():
    if not qdrant_collection_exists():
        if not WORKER_MAY_CREATE_QDRANT_COLLECTION:
            raise RuntimeError(
                f"Shared Qdrant collection {QDRANT_COLLECTION!r} does not exist at {QDRANT_URL}. "
                "Initialize the build collection on the service node before starting workers."
            )

        # Development/bootstrap escape hatch only. Normal concurrent waves should
        # initialize the collection centrally before any worker starts.
        model = load_embedding_model()
        dimension = int(model.get_sentence_embedding_dimension())
        try:
            get_qdrant_client().create_collection(
                collection_name=QDRANT_COLLECTION,
                vectors_config=VectorParams(size=dimension, distance=Distance.COSINE),
            )
        except Exception as exc:
            if not qdrant_collection_exists():
                raise
            print("Collection creation raced with another worker; continuing:", exc)

    model = load_embedding_model()
    expected_dimension = int(model.get_sentence_embedding_dimension())
    existing_dimension = _existing_collection_vector_size()
    if existing_dimension is not None and existing_dimension != expected_dimension:
        raise RuntimeError(
            f"Qdrant vector-size mismatch: collection={existing_dimension}, "
            f"embedding model={expected_dimension}."
        )

    ensure_payload_indexes()
    print(
        "Shared Qdrant collection validated:",
        QDRANT_COLLECTION,
        "points=",
        qdrant_point_count(),
    )

def qdrant_point_exists(point_id: str) -> bool:
    if not qdrant_collection_exists():
        return False
    points = get_qdrant_client().retrieve(
        collection_name=QDRANT_COLLECTION,
        ids=[point_id],
        with_payload=False,
        with_vectors=False,
    )
    return bool(points)

print("Shared Qdrant validation helpers ready.")


## 15. Batch embedding + coordinator-backed global chunk dedupe + direct Qdrant upsert

RAG chunk text remains globally deduplicated across the build. Before embedding, the worker atomically claims each raw chunk `content_hash` through the coordinator. Claimed chunks are embedded locally and upserted directly to Qdrant with deterministic UUID5 point IDs.


In [ ]:
def reconstruct_rag_chunk_text(row: sqlite3.Row) -> str:
    specs = build_rag_chunk_specs(row["document_id"])
    for spec in specs:
        if int(spec["page"]) == int(row["page"]) and int(spec["chunk_index"]) == int(row["chunk_index"]):
            return spec["text"]
    raise KeyError(
        f"Cannot reconstruct RAG chunk {row['document_id']} page={row['page']} "
        f"chunk={row['chunk_index']}"
    )

def _mark_rag_indexed_recovered(row: sqlite3.Row):
    with get_db() as conn:
        conn.execute(
            """
            UPDATE rag_chunks
            SET status='indexed',
                embedding_status='embedded',
                qdrant_status='indexed',
                failure_stage=NULL,
                last_error=NULL,
                updated_at=?
            WHERE rag_chunk_id=?
            """,
            (utc_now(), row["rag_chunk_id"]),
        )

def _mark_rag_duplicate(row: sqlite3.Row):
    with get_db() as conn:
        conn.execute(
            """
            UPDATE rag_chunks
            SET status='duplicate_skipped',
                embedding_status='skipped',
                qdrant_status='duplicate_skipped',
                failure_stage=NULL,
                last_error=NULL,
                updated_at=?
            WHERE rag_chunk_id=?
            """,
            (utc_now(), row["rag_chunk_id"]),
        )

def _retry_metadata_if_needed(row: sqlite3.Row) -> Tuple[Dict[str, Any], Dict[str, Any], bool]:
    """Rebuild metadata on metadata-stage retries so updated mappings/config can fix a failure."""
    payload = json.loads(row["metadata_json"] or "{}")
    is_retry = row["status"] == "failed" and row["failure_stage"] == "metadata"

    if is_retry:
        chunk_text = reconstruct_rag_chunk_text(row)
        payload = enrich_chunk_metadata(
            row["document_id"], int(row["page"]), int(row["chunk_index"]), chunk_text
        )

    validation = validate_qdrant_metadata(payload)
    if not is_retry:
        return payload, validation, validation["valid"]

    attempt_number = int(row["metadata_attempt_count"]) + 1
    started = utc_now()
    with get_db() as conn:
        conn.execute(
            """
            UPDATE rag_chunks
            SET metadata_json=?,
                metadata_attempt_count=metadata_attempt_count+1,
                status=?,
                failure_stage=?,
                last_error=?,
                updated_at=?
            WHERE rag_chunk_id=?
            """,
            (
                json.dumps(payload, ensure_ascii=False),
                "metadata_validated" if validation["valid"] else "failed",
                None if validation["valid"] else "metadata",
                None if validation["valid"] else json.dumps(validation),
                utc_now(), row["rag_chunk_id"],
            ),
        )
    record_attempt(
        "rag_chunk", row["rag_chunk_id"], "metadata", attempt_number,
        "succeeded" if validation["valid"] else "failed",
        started,
        None if validation["valid"] else ValueError(json.dumps(validation)),
    )
    return payload, validation, validation["valid"]

def process_rag_batch(rows: Sequence[sqlite3.Row]):
    if not rows:
        return

    assert_state_lease_healthy()
    model = load_embedding_model()
    client = get_qdrant_client()

    work_rows: List[sqlite3.Row] = []
    texts: List[str] = []
    payloads: List[Dict[str, Any]] = []
    claimed_hashes: List[str] = []
    batch_hashes: Set[str] = set()

    for row in rows:
        payload, validation, valid = _retry_metadata_if_needed(row)
        if not valid:
            continue

        chunk_hash = payload["content_hash"]
        if chunk_hash in batch_hashes:
            _mark_rag_duplicate(row)
            continue

        try:
            claim = coordinator_claim_content(
                "rag_chunk",
                chunk_hash,
                resource_id=row["qdrant_point_id"],
            )
        except Exception as exc:
            with get_db() as conn:
                conn.execute(
                    """
                    UPDATE rag_chunks
                    SET status='failed', failure_stage='coordinator_claim',
                        last_error=?, updated_at=?
                    WHERE rag_chunk_id=?
                    """,
                    (traceback.format_exc()[-4000:], utc_now(), row["rag_chunk_id"]),
                )
            record_attempt(
                "rag_chunk", row["rag_chunk_id"], "coordinator_claim",
                int(row["qdrant_attempt_count"]) + 1, "failed", utc_now(), exc,
            )
            continue

        claim_status = _normalized_status(claim)
        claimed_resource = str(claim.get("resource_id") or "")
        owner_state = str(claim.get("owner_state") or "").upper()

        if claim_status == "already_complete":
            if claimed_resource == row["qdrant_point_id"]:
                # Crash-recovery path: coordinator completion was committed before
                # the local ledger recorded success. Verify Qdrant before trusting it.
                if not qdrant_point_exists(row["qdrant_point_id"]):
                    raise RuntimeError(
                        "Coordinator says RAG chunk is complete but its Qdrant point is missing: "
                        f"{row['qdrant_point_id']}"
                    )
                _mark_rag_indexed_recovered(row)
            else:
                _mark_rag_duplicate(row)
            continue

        if claim_status == "claimed_by_other":
            if owner_state == STATE_CODE:
                with get_db() as conn:
                    conn.execute(
                        """
                        UPDATE rag_chunks
                        SET status='failed', failure_stage='coordinator_claim',
                            last_error=?, updated_at=?
                        WHERE rag_chunk_id=?
                        """,
                        (
                            "Same-state RAG claim is still owned by another worker; "
                            "coordinator must recover/transfer stale claim",
                            utc_now(), row["rag_chunk_id"],
                        ),
                    )
                continue
            _mark_rag_duplicate(row)
            continue

        if claim_status not in {"claimed", "already_owned"}:
            raise RuntimeError(f"Unexpected RAG chunk claim response: {claim}")

        batch_hashes.add(chunk_hash)
        claimed_hashes.append(chunk_hash)
        work_rows.append(row)
        texts.append(payload["text"])
        payloads.append(payload)

    if not work_rows:
        return

    embed_started = utc_now()
    try:
        vectors = model.encode(
            texts,
            batch_size=min(EMBED_BATCH_SIZE, len(texts)),
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False,
        )
    except Exception as exc:
        with get_db() as conn:
            for row in work_rows:
                conn.execute(
                    """
                    UPDATE rag_chunks
                    SET embedding_status='failed',
                        embedding_attempt_count=embedding_attempt_count+1,
                        status='failed',
                        failure_stage='embed',
                        last_error=?,
                        updated_at=?
                    WHERE rag_chunk_id=?
                    """,
                    (traceback.format_exc()[-4000:], utc_now(), row["rag_chunk_id"]),
                )
        for row, chunk_hash in zip(work_rows, claimed_hashes):
            record_attempt(
                "rag_chunk", row["rag_chunk_id"], "embed",
                int(row["embedding_attempt_count"]) + 1,
                "failed", embed_started, exc,
            )
            coordinator_release_content("rag_chunk", chunk_hash, "embedding_failed")
        return

    with get_db() as conn:
        for row in work_rows:
            conn.execute(
                """
                UPDATE rag_chunks
                SET embedding_status='embedded',
                    embedding_attempt_count=embedding_attempt_count+1,
                    failure_stage=NULL,
                    last_error=NULL,
                    updated_at=?
                WHERE rag_chunk_id=?
                """,
                (utc_now(), row["rag_chunk_id"]),
            )
    for row in work_rows:
        record_attempt(
            "rag_chunk", row["rag_chunk_id"], "embed",
            int(row["embedding_attempt_count"]) + 1,
            "succeeded", embed_started,
        )

    # Refresh short-lived chunk claims after the potentially expensive embedding step.
    try:
        for chunk_hash in claimed_hashes:
            heartbeat = coordinator_heartbeat_content("rag_chunk", chunk_hash)
            if _normalized_status(heartbeat) not in {
                "renewed", "active", "already_owned", "claimed"
            }:
                raise RuntimeError(f"RAG chunk claim lease lost: {heartbeat}")
    except Exception as exc:
        with get_db() as conn:
            for row in work_rows:
                conn.execute(
                    """
                    UPDATE rag_chunks
                    SET status='failed', failure_stage='coordinator_claim',
                        last_error=?, updated_at=?
                    WHERE rag_chunk_id=?
                    """,
                    (traceback.format_exc()[-4000:], utc_now(), row["rag_chunk_id"]),
                )
        for chunk_hash in claimed_hashes:
            coordinator_release_content("rag_chunk", chunk_hash, "claim_heartbeat_failed")
        return

    upsert_started = utc_now()
    try:
        points = [
            PointStruct(
                id=row["qdrant_point_id"],
                vector=vector.tolist(),
                payload=payload,
            )
            for row, vector, payload in zip(work_rows, vectors, payloads)
        ]

        client.upsert(
            collection_name=QDRANT_COLLECTION,
            points=points,
            wait=True,
        )
    except Exception as exc:
        with get_db() as conn:
            for row in work_rows:
                conn.execute(
                    """
                    UPDATE rag_chunks
                    SET qdrant_status='failed',
                        qdrant_attempt_count=qdrant_attempt_count+1,
                        status='failed',
                        failure_stage='qdrant_upsert',
                        last_error=?,
                        updated_at=?
                    WHERE rag_chunk_id=?
                    """,
                    (traceback.format_exc()[-4000:], utc_now(), row["rag_chunk_id"]),
                )
        for row, chunk_hash in zip(work_rows, claimed_hashes):
            record_attempt(
                "rag_chunk", row["rag_chunk_id"], "qdrant_upsert",
                int(row["qdrant_attempt_count"]) + 1,
                "failed", upsert_started, exc,
            )
            coordinator_release_content("rag_chunk", chunk_hash, "qdrant_upsert_failed")
        return

    # Complete coordinator claims before recording the local indexed state. If a
    # response is lost after Qdrant committed, deterministic IDs + claim resource_id
    # let the next retry reconcile rather than create a duplicate.
    try:
        for row, chunk_hash in zip(work_rows, claimed_hashes):
            coordinator_complete_content(
                "rag_chunk",
                chunk_hash,
                resource_id=row["qdrant_point_id"],
            )
    except Exception as exc:
        with get_db() as conn:
            for row in work_rows:
                conn.execute(
                    """
                    UPDATE rag_chunks
                    SET qdrant_status='indexed',
                        qdrant_attempt_count=qdrant_attempt_count+1,
                        status='failed',
                        failure_stage='coordinator_complete',
                        last_error=?,
                        updated_at=?
                    WHERE rag_chunk_id=?
                    """,
                    (traceback.format_exc()[-4000:], utc_now(), row["rag_chunk_id"]),
                )
        for row in work_rows:
            record_attempt(
                "rag_chunk", row["rag_chunk_id"], "coordinator_complete",
                int(row["qdrant_attempt_count"]) + 1,
                "failed", upsert_started, exc,
            )
        return

    with get_db() as conn:
        for row in work_rows:
            conn.execute(
                """
                UPDATE rag_chunks
                SET qdrant_status='indexed',
                    qdrant_attempt_count=qdrant_attempt_count+1,
                    status='indexed',
                    failure_stage=NULL,
                    last_error=NULL,
                    updated_at=?
                WHERE rag_chunk_id=?
                """,
                (utc_now(), row["rag_chunk_id"]),
            )
    for row in work_rows:
        record_attempt(
            "rag_chunk", row["rag_chunk_id"], "qdrant_upsert",
            int(row["qdrant_attempt_count"]) + 1,
            "succeeded", upsert_started,
        )

def rag_index_pass(statuses: Sequence[str] = ("metadata_validated",)):
    placeholders = ",".join("?" for _ in statuses)
    with get_db() as conn:
        rows = conn.execute(
            f"""
            SELECT * FROM rag_chunks
            WHERE run_id=? AND status IN ({placeholders})
            ORDER BY document_id, page, chunk_index
            """,
            (RUN_ID, *statuses),
        ).fetchall()

    if not rows:
        print("No RAG chunks for this indexing pass.")
        return

    for start in tqdm(
        range(0, len(rows), QDRANT_UPSERT_BATCH_SIZE),
        desc="Embedding + Qdrant batches",
    ):
        batch = rows[start:start + QDRANT_UPSERT_BATCH_SIZE]
        process_rag_batch(batch)

def retry_failed_rag():
    for retry_idx in range(MAX_STAGE_RETRIES):
        with get_db() as conn:
            failed = conn.execute(
                """
                SELECT COUNT(*) FROM rag_chunks
                WHERE run_id=? AND status='failed'
                """,
                (RUN_ID,),
            ).fetchone()[0]

        if failed == 0:
            break

        print(f"RAG retry {retry_idx + 1}/{MAX_STAGE_RETRIES}: {failed} chunks")
        rag_index_pass(("failed",))

    with get_db() as conn:
        conn.execute(
            """
            UPDATE rag_chunks
            SET status='permanently_failed',
                qdrant_status=CASE
                    WHEN qdrant_status='indexed' THEN qdrant_status
                    ELSE 'permanently_failed'
                END,
                updated_at=?
            WHERE run_id=? AND status='failed'
            """,
            (utc_now(), RUN_ID),
        )

def finalize_document_rag_states():
    with get_db() as conn:
        docs = conn.execute(
            """
            SELECT document_id
            FROM run_documents
            WHERE run_id=? AND accepted=1
            """,
            (RUN_ID,),
        ).fetchall()

    for row in docs:
        doc_id = row["document_id"]
        with get_db() as conn:
            counts = {
                status: count
                for status, count in conn.execute(
                    """
                    SELECT status, COUNT(*)
                    FROM rag_chunks
                    WHERE run_id=? AND document_id=?
                    GROUP BY status
                    """,
                    (RUN_ID, doc_id),
                ).fetchall()
            }

        nonterminal = sum(
            counts.get(s, 0)
            for s in [
                "pending", "metadata_enriching", "metadata_validated",
                "embedding", "embedded", "qdrant_pending", "failed",
            ]
        )
        if nonterminal:
            continue

        terminal_success = counts.get("indexed", 0) + counts.get("duplicate_skipped", 0)
        perm_failed = counts.get("permanently_failed", 0)

        if terminal_success == 0 and perm_failed > 0:
            doc_status = "permanently_failed"
            rag_status = "permanently_failed"
            qdrant_status = "permanently_failed"
        else:
            doc_status = "indexed"
            rag_status = "complete"
            qdrant_status = "indexed"

        with get_db() as conn:
            conn.execute(
                """
                UPDATE run_documents
                SET document_status=?, rag_status=?, qdrant_status=?, updated_at=?
                WHERE run_id=? AND document_id=?
                """,
                (doc_status, rag_status, qdrant_status, utc_now(), RUN_ID, doc_id),
            )

print("Batch embedding/Qdrant pipeline ready.")

## 16. State terminal validation, statistics, state crop artifact, and state manifest

A worker finalizes **only its own state**. It does not update the cumulative global crop JSON and does not create a Qdrant snapshot. The separate wave finalizer owns all build-level global mutations/checkpoints.


In [ ]:
SOURCE_TERMINAL = {"extracted", "duplicate_skipped", "permanently_failed"}
DOC_TERMINAL = {"indexed", "rejected", "duplicate_skipped", "permanently_failed"}
Q_TERMINAL = {"succeeded", "skipped_early_stop", "permanently_failed"}
RAG_TERMINAL = {"indexed", "duplicate_skipped", "permanently_failed"}

def table_status_counts(table: str, status_field: str) -> Dict[str, int]:
    with get_db() as conn:
        rows = conn.execute(
            f"""
            SELECT {status_field}, COUNT(*) AS n
            FROM {table}
            WHERE run_id=?
            GROUP BY {status_field}
            """,
            (RUN_ID,),
        ).fetchall()
    return {row[0]: int(row[1]) for row in rows}

def assert_all_terminal():
    problems = []

    source_counts = table_status_counts("source_tasks", "status")
    doc_counts = table_status_counts("run_documents", "document_status")
    q_counts = table_status_counts("qualification_chunks", "status")
    rag_counts = table_status_counts("rag_chunks", "status")

    for name, counts, terminal in [
        ("source_tasks", source_counts, SOURCE_TERMINAL),
        ("run_documents", doc_counts, DOC_TERMINAL),
        ("qualification_chunks", q_counts, Q_TERMINAL),
        ("rag_chunks", rag_counts, RAG_TERMINAL),
    ]:
        nonterminal = {k: v for k, v in counts.items() if k not in terminal}
        if nonterminal:
            problems.append(f"{name}: {nonterminal}")

    if problems:
        raise RuntimeError("Non-terminal work remains:\n" + "\n".join(problems))

    return {
        "source_tasks": source_counts,
        "run_documents": doc_counts,
        "qualification_chunks": q_counts,
        "rag_chunks": rag_counts,
    }

def metadata_quality_stats() -> Dict[str, Any]:
    with get_db() as conn:
        rows = conn.execute(
            """
            SELECT status, metadata_json, failure_stage
            FROM rag_chunks
            WHERE run_id=?
            """,
            (RUN_ID,),
        ).fetchall()

    stats = Counter()
    missing_or_invalid = Counter()

    for row in rows:
        stats["rag_chunks_total"] += 1
        payload = json.loads(row["metadata_json"] or "{}")
        validation = validate_qdrant_metadata(payload)

        if validation["valid"]:
            stats["contract_valid"] += 1
        elif row["status"] == "permanently_failed" and row["failure_stage"] == "metadata":
            stats["contract_permanently_failed"] += 1

        if payload.get("location"):
            stats["location_present"] += 1
        if payload.get("hardiness_zone"):
            stats["hardiness_zone_present"] += 1
        if validation["ak_hi_hardiness_exception"]:
            stats["hardiness_zone_ak_hi_exception"] += 1
        if payload.get("title"):
            stats["title_present"] += 1

        if "month_year" in payload:
            stats["month_year_field_present"] += 1
        if payload.get("month_year"):
            stats["month_year_known"] += 1
        else:
            stats["month_year_unknown"] += 1

        if payload.get("title") and payload.get("month_year") and payload.get("hardiness_zone"):
            stats["priority_full_metadata"] += 1

        for field in validation["missing_fields"] + validation["invalid_fields"]:
            missing_or_invalid[field] += 1

    expected_stat_keys = [
        "rag_chunks_total",
        "contract_valid",
        "contract_permanently_failed",
        "location_present",
        "hardiness_zone_present",
        "hardiness_zone_ak_hi_exception",
        "title_present",
        "month_year_field_present",
        "month_year_known",
        "month_year_unknown",
        "priority_full_metadata",
    ]
    result = {key: int(stats.get(key, 0)) for key in expected_stat_keys}
    result["missing_or_invalid"] = {
        field: int(missing_or_invalid.get(field, 0))
        for field in REQUIRED_QDRANT_FIELDS
    }
    return result

def get_qdrant_version() -> str:
    try:
        r = requests.get(
            QDRANT_URL.rstrip("/") + "/",
            headers=qdrant_headers(),
            timeout=10,
        )
        if r.ok:
            data = r.json()
            return str(data.get("version") or data.get("title") or "")
    except Exception:
        pass
    return ""

def run_validation_queries() -> List[Dict[str, Any]]:
    if not VALIDATION_QUERIES:
        return []

    model = load_embedding_model()
    client = get_qdrant_client()
    results = []

    for query in VALIDATION_QUERIES:
        vector = model.encode(
            [query],
            show_progress_bar=False,
            convert_to_numpy=True,
            normalize_embeddings=False,
        )[0].tolist()

        points = client.query_points(
            collection_name=QDRANT_COLLECTION,
            query=vector,
            limit=3,
            with_payload=True,
        ).points

        results.append({
            "query": query,
            "hits": len(points),
            "top_titles": [
                (p.payload or {}).get("title", "")
                for p in points
            ],
        })

    return results

def file_sha256(path: Path) -> str:
    return bytes_sha256(path)

def current_input_fingerprints() -> Dict[str, Any]:
    def fingerprint(path: Optional[Path]) -> Optional[Dict[str, Any]]:
        if path is None:
            return None
        path = Path(path)
        return {
            "name": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": file_sha256(path),
        }

    return {
        "pdf_zip": fingerprint(PDF_ZIP_FILE),
        "csv_zip": fingerprint(CSV_ZIP_FILE),
        "url_file": fingerprint(URL_FILE),
        "hardiness_csv": fingerprint(HARDINESS_CSV),
        "global_crop_seed": fingerprint(GLOBAL_CROP_OCCURRENCE_JSON),
    }

def bind_or_validate_input_fingerprints() -> Dict[str, Any]:
    fingerprints = current_input_fingerprints()
    serialized = json.dumps(fingerprints, sort_keys=True, ensure_ascii=False)

    with get_db() as conn:
        row = conn.execute(
            "SELECT input_fingerprint_json FROM runs WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()
        existing = row["input_fingerprint_json"] if row else None

        if existing:
            if json.loads(existing) != fingerprints:
                raise RuntimeError(
                    "Input/support fingerprints changed for an existing state run. "
                    "Do not resume a state with different inputs. Restore the original "
                    "worker inputs/support artifacts or use a corrected/new BUILD_ID."
                )
        else:
            conn.execute(
                "UPDATE runs SET input_fingerprint_json=? WHERE run_id=?",
                (serialized, RUN_ID),
            )

    return fingerprints

def state_statistics() -> Dict[str, Any]:
    with get_db() as conn:
        source_total = conn.execute(
            "SELECT COUNT(*) FROM source_tasks WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()[0]
        duplicate_count = conn.execute(
            "SELECT COUNT(*) FROM source_tasks WHERE run_id=? AND status='duplicate_skipped'",
            (RUN_ID,),
        ).fetchone()[0]
        source_perm_failed = conn.execute(
            "SELECT COUNT(*) FROM source_tasks WHERE run_id=? AND status='permanently_failed'",
            (RUN_ID,),
        ).fetchone()[0]

        docs = conn.execute(
            """
            SELECT
              SUM(CASE WHEN accepted=1 THEN 1 ELSE 0 END) AS accepted,
              SUM(CASE WHEN qualification_tag='msc' THEN 1 ELSE 0 END) AS rejected,
              SUM(CASE WHEN document_status='permanently_failed' THEN 1 ELSE 0 END) AS perm_failed
            FROM run_documents
            WHERE run_id=? AND duplicate=0
            """,
            (RUN_ID,),
        ).fetchone()

        q_total = conn.execute(
            "SELECT COUNT(*) FROM qualification_chunks WHERE run_id=? AND status='succeeded'",
            (RUN_ID,),
        ).fetchone()[0]
        q_perm = conn.execute(
            "SELECT COUNT(*) FROM qualification_chunks WHERE run_id=? AND status='permanently_failed'",
            (RUN_ID,),
        ).fetchone()[0]
        rag_created = conn.execute(
            "SELECT COUNT(*) FROM rag_chunks WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()[0]
        rag_indexed = conn.execute(
            "SELECT COUNT(*) FROM rag_chunks WHERE run_id=? AND status='indexed'",
            (RUN_ID,),
        ).fetchone()[0]
        rag_duplicate = conn.execute(
            "SELECT COUNT(*) FROM rag_chunks WHERE run_id=? AND status='duplicate_skipped'",
            (RUN_ID,),
        ).fetchone()[0]
        rag_perm = conn.execute(
            "SELECT COUNT(*) FROM rag_chunks WHERE run_id=? AND status='permanently_failed'",
            (RUN_ID,),
        ).fetchone()[0]

    return {
        "sources_discovered": int(source_total or 0),
        "duplicates_skipped": int(duplicate_count or 0),
        "sources_permanently_failed": int(source_perm_failed or 0),
        "documents_accepted": int(docs["accepted"] or 0),
        "documents_rejected": int(docs["rejected"] or 0),
        "documents_permanently_failed": int(docs["perm_failed"] or 0),
        "qualification_chunks_processed": int(q_total or 0),
        "qualification_chunks_permanently_failed": int(q_perm or 0),
        "rag_chunks_created": int(rag_created or 0),
        "rag_chunks_indexed": int(rag_indexed or 0),
        "rag_chunks_duplicate_skipped": int(rag_duplicate or 0),
        "rag_chunks_permanently_failed": int(rag_perm or 0),
        "qdrant_points_observed_at_state_finalize": qdrant_point_count(),
    }

def finalize_state_artifacts(input_fingerprints: Dict[str, Any]):
    terminal_counts = assert_all_terminal()

    with get_db() as conn:
        conn.execute(
            "UPDATE runs SET status='validating' WHERE run_id=?",
            (RUN_ID,),
        )

    metadata_quality = metadata_quality_stats()
    crop_dictionary = build_crop_dictionary_from_ledger()
    crop_path = write_state_crop_occurrences(crop_dictionary)
    stats = state_statistics()

    with get_db() as conn:
        run_row = conn.execute(
            "SELECT started_at FROM runs WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()

    manifest = {
        "schema_version": "2.0",
        "artifact_type": "concurrent_preload_state_manifest",
        "status": "complete",
        "pipeline_version": WORKER_PIPELINE_VERSION,
        "build_id": BUILD_ID,
        "wave_id": WAVE_ID,
        "run_id": RUN_ID,
        "state_processed": {
            "name": STATE_NAME,
            "code": STATE_CODE,
            "state_key": STATE_KEY,
        },
        "worker": {
            "slot": WORKER_SLOT,
            "worker_id": WORKER_ID,
            "hostname": socket.gethostname(),
        },
        "services": {
            "coordinator_url": COORDINATOR_URL,
            "qdrant_url": QDRANT_URL,
            "qdrant_collection": QDRANT_COLLECTION,
        },
        "versions": {
            "extractor": EXTRACTOR_VERSION,
            "classifier": CLASSIFIER_VERSION,
            "chunker": CHUNKER_VERSION,
            "metadata_contract": METADATA_CONTRACT_VERSION,
            "classifier_model": CLASSIFIER_MODEL_ID,
            "embedding_model": EMBEDDING_MODEL,
            "qdrant": get_qdrant_version(),
        },
        "chunk_identity": {
            "logical_id": "source_id|page|chunk_index|content_hash",
            "qdrant_point_id": "UUID5(logical_id)",
        },
        "input_fingerprints": input_fingerprints,
        "state_statistics": stats,
        "metadata_quality": metadata_quality,
        "terminal_status_counts": terminal_counts,
        "crop_dictionary": {
            "state_file": str(crop_path),
            "sha256": file_sha256(crop_path),
            "global_seed_file": str(GLOBAL_CROP_OCCURRENCE_JSON),
            "mode": "state_local_output_for_wave_merge",
            "crop_count": len(crop_dictionary.get(STATE_KEY, {})),
        },
        "started_at": run_row["started_at"] if run_row else None,
        "completed_at": utc_now(),
    }

    # Write the final state manifest first, then atomically mark the state complete
    # in the coordinator using this exact file hash. finalize_wave.ipynb must require
    # BOTH the complete manifest and coordinator COMPLETE status, so a crash in
    # between cannot prematurely finalize the wave.
    write_json_atomic(STATE_MANIFEST_PATH, manifest)

    manifest_sha = file_sha256(STATE_MANIFEST_PATH)
    coordinator_mark_state_complete(STATE_MANIFEST_PATH, manifest_sha)

    with get_db() as conn:
        conn.execute(
            """
            UPDATE runs
            SET status='complete', completed_at=?, manifest_path=?, error=NULL
            WHERE run_id=?
            """,
            (manifest["completed_at"], str(STATE_MANIFEST_PATH), RUN_ID),
        )

    print("State finalized.")
    print("State manifest:", STATE_MANIFEST_PATH)
    print("State crop artifact:", crop_path)
    print("No Qdrant snapshot was created by this worker.")
    return manifest

print("State validation/manifest helpers ready.")


## 17. End-to-end concurrent worker orchestrator

The worker is resume-safe. It acquires one state lease, validates that its inputs have not changed, uses coordinator-backed global dedupe, writes directly to shared Qdrant, and finalizes only state-local artifacts.


In [ ]:
def preflight_environment():
    configured_sources = bool(URL_FILE or PDF_DIR or PDF_ZIP_FILE or CSV_INPUTS)
    if not configured_sources:
        raise RuntimeError(
            f"No input sources are configured under {INPUT_DIR} for this state run."
        )

    try:
        response = requests.get(
            QDRANT_URL.rstrip("/") + "/collections",
            headers=qdrant_headers(),
            timeout=10,
        )
        response.raise_for_status()
    except Exception as exc:
        raise RuntimeError(
            f"Qdrant is not reachable at {QDRANT_URL}. The shared service-node "
            "Qdrant server must be running before launching a worker."
        ) from exc

    try:
        health = coordinator_health()
    except Exception as exc:
        raise RuntimeError(
            f"Preload Coordinator is not reachable at {COORDINATOR_URL}."
        ) from exc

    print("Preflight OK: inputs configured, Qdrant reachable, coordinator reachable.")
    print("Coordinator health:", health)

def print_run_status():
    with get_db() as conn:
        run = conn.execute(
            "SELECT * FROM runs WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()

    print(f"Run {RUN_ID}: {run['status']}")
    print("Build/wave/state:", BUILD_ID, WAVE_ID, STATE_CODE)
    for table, field in [
        ("source_tasks", "status"),
        ("run_documents", "document_status"),
        ("qualification_chunks", "status"),
        ("rag_chunks", "status"),
    ]:
        try:
            print(f"{table}: {table_status_counts(table, field)}")
        except Exception as exc:
            print(f"{table}: {exc}")

def run_pipeline():
    preflight_environment()
    lease = coordinator_acquire_state()

    with get_db() as conn:
        existing_run = conn.execute(
            "SELECT status, manifest_path FROM runs WHERE run_id=?",
            (RUN_ID,),
        ).fetchone()

    if _STATE_ALREADY_COMPLETE:
        manifest_path = Path(
            (existing_run["manifest_path"] if existing_run else None)
            or STATE_MANIFEST_PATH
        )
        manifest = load_json(manifest_path, None) if manifest_path.exists() else None
        if (
            isinstance(manifest, dict)
            and manifest.get("status") == "complete"
            and manifest.get("build_id") == BUILD_ID
            and manifest.get("wave_id") == WAVE_ID
            and (manifest.get("state_processed") or {}).get("code") == STATE_CODE
        ):
            # Crash-recovery window: coordinator may have committed COMPLETE before
            # the final local runs-row update. Reconcile the local ledger from the
            # already-final manifest rather than reprocessing the state.
            completed_at = manifest.get("completed_at") or utc_now()
            with get_db() as conn:
                conn.execute(
                    """
                    UPDATE runs
                    SET status='complete', completed_at=?, manifest_path=?, error=NULL
                    WHERE run_id=?
                    """,
                    (completed_at, str(manifest_path), RUN_ID),
                )
            print(
                f"State {STATE_CODE} is already complete in the coordinator; "
                "local completion state reconciled from the existing manifest."
            )
            return manifest
        raise RuntimeError(
            "Coordinator says this state is already complete but no matching complete "
            "state manifest is available locally. Reconcile the durable state directory "
            "before doing any reprocessing."
        )

    start_state_heartbeat()

    try:
        input_fingerprints = bind_or_validate_input_fingerprints()
        assert_state_lease_healthy()

        with get_db() as conn:
            conn.execute(
                """
                UPDATE runs
                SET status='running',
                    started_at=COALESCE(started_at, ?),
                    worker_id=?, error=NULL
                WHERE run_id=?
                """,
                (utc_now(), WORKER_ID, RUN_ID),
            )

        print("\n=== 1/8 Discover sources ===")
        discover_sources()
        assert_state_lease_healthy()

        print("\n=== 2/8 Extract + canonicalize + global document claims ===")
        extraction_pass(("discovered", "failed"))
        retry_failed_extraction()
        assert_state_lease_healthy()

        print("\n=== 3/8 Qualification initial pass ===")
        qualification_initial_pass()
        assert_state_lease_healthy()

        print("\n=== 4/8 Qualification retries + decisions ===")
        with get_db() as conn:
            conn.execute(
                "UPDATE runs SET status='retrying' WHERE run_id=?",
                (RUN_ID,),
            )
        retry_failed_qualification()
        finalize_qualification_decisions()
        assert_state_lease_healthy()

        # Qualification model is no longer needed; free GPU before embeddings.
        unload_classifier_model()

        print("\n=== 5/8 Prepare RAG chunks + metadata contract ===")
        prepare_rag_chunks()
        assert_state_lease_healthy()

        print("\n=== 6/8 Validate shared Qdrant + global chunk claims + batch indexing ===")
        validate_shared_qdrant_collection()
        rag_index_pass(("metadata_validated",))
        assert_state_lease_healthy()

        print("\n=== 7/8 RAG retries + document terminal states ===")
        retry_failed_rag()
        finalize_document_rag_states()
        assert_state_lease_healthy()

        print("\n=== 8/8 Validate state + state crop artifact + state manifest ===")
        manifest = finalize_state_artifacts(input_fingerprints)

        stop_state_heartbeat()
        print("\n✓ STATE COMPLETE:", RUN_ID)
        print("Wave snapshot/global crop merge must be performed by finalize_wave.ipynb.")
        return manifest

    except Exception as exc:
        stop_state_heartbeat()
        with get_db() as conn:
            conn.execute(
                "UPDATE runs SET status='failed', error=? WHERE run_id=?",
                (traceback.format_exc()[-8000:], RUN_ID),
            )
        try:
            coordinator_release_state(reason=f"worker_failed:{type(exc).__name__}")
        except Exception as release_exc:
            print("Coordinator state-release warning:", release_exc)
        print_run_status()
        raise

print("Concurrent worker orchestrator ready.")


## 18. Review configuration, then run

`RUN_PIPELINE` defaults to `False`. Before enabling it:

1. confirm `BUILD_ID`, `WAVE_ID`, `STATE_NAME`, and `STATE_CODE`;
2. replace only this worker's `input/` contents;
3. confirm the service-node coordinator and Qdrant endpoints;
4. verify the shared Qdrant build collection already exists;
5. keep `RUN_MODE="resume"` for ordinary restarts.


## Operational safety notes

- Keep `RUN_PIPELINE = False` until the worker assignment, input folder, coordinator endpoint, and Qdrant endpoint are verified.
- The shared Qdrant collection must already exist before normal concurrent workers start.
- Never run two workers for the same state/build concurrently; the coordinator lease is the enforcement mechanism.
- Do not change a state's inputs while resuming it. Stored input fingerprints intentionally block mixed-input resumes.
- A worker failure never restores or rolls back shared Qdrant. Resume the failed state from its state-local durable directory.
- Do not run the wave finalizer until every explicitly expected state is complete.


In [ ]:
print_run_status()

if RUN_PIPELINE:
    manifest = run_pipeline()
else:
    print(
        "\nRUN_PIPELINE is False. Review the configuration cell, "
        "set RUN_PIPELINE = True, and rerun this cell when this worker assignment is ready."
    )

## 19. Recovery / inspection helpers

These helpers inspect this state's private ledger. Normal failures are resumed by rerunning the same state with the same `BUILD_ID`, `WAVE_ID`, and inputs. The shared Qdrant collection is never rolled back by a worker.


In [ ]:
def show_failed_units(limit: int = 50):
    with get_db() as conn:
        sources = conn.execute(
            """
            SELECT source_key, source_type, source_uri, status, attempt_count, last_error
            FROM source_tasks
            WHERE run_id=? AND status IN ('failed','permanently_failed')
            LIMIT ?
            """,
            (RUN_ID, limit),
        ).fetchall()

        qchunks = conn.execute(
            """
            SELECT qualification_chunk_id, document_id, chunk_index,
                   status, attempt_count, last_error
            FROM qualification_chunks
            WHERE run_id=? AND status IN ('failed','permanently_failed')
            LIMIT ?
            """,
            (RUN_ID, limit),
        ).fetchall()

        rchunks = conn.execute(
            """
            SELECT rag_chunk_id, document_id, page, chunk_index,
                   status, failure_stage, metadata_attempt_count,
                   embedding_attempt_count, qdrant_attempt_count, last_error
            FROM rag_chunks
            WHERE run_id=? AND status IN ('failed','permanently_failed')
            LIMIT ?
            """,
            (RUN_ID, limit),
        ).fetchall()

    print("Source failures:")
    for row in sources:
        print(dict(row))

    print("\nQualification failures:")
    for row in qchunks:
        print(dict(row))

    print("\nRAG failures:")
    for row in rchunks:
        print(dict(row))

def show_metadata_quality():
    print(json.dumps(metadata_quality_stats(), indent=2))

def show_run_manifest():
    path = STATE_DIR / "manifest.json"
    if not path.exists():
        print("Manifest has not been created yet.")
        return
    print(path.read_text(encoding="utf-8"))

print("Inspection helpers ready.")

## 20. Concurrent-worker safety notes

- **One state per worker notebook.** Multiple different states may run concurrently.
- The coordinator prevents two active workers from owning the same `(BUILD_ID, STATE_CODE)`.
- State persistence is under `persistent_state/<BUILD_ID>/<STATE_CODE>/`, so generic worker folders can be reused across waves.
- Workers read but never modify the cumulative global `crop_occurrences.json`.
- Workers emit `crop_occurrences_state.json`; `finalize_wave.ipynb` merges those state outputs only after all explicitly expected states are complete.
- Qdrant point IDs use UUID5 over the agreed logical identity `source_id + page + chunk_index + content_hash`.
- Whole-document and RAG-chunk content hashes are claimed atomically through the coordinator to preserve strict build-level deduplication.
- Workers write vectors directly to one shared Qdrant server.
- Workers never reset, restore, delete, or snapshot the shared Qdrant collection.
- A crashed worker resumes from its own ledger. Other states remain untouched.
- If a state was processed with semantically wrong inputs/configuration, do not surgically delete it from the shared build in v1; use a corrected/new build.
- Wave size is variable. Five generic workers is a practical maximum, not a correctness requirement.
- The separate `finalize_wave.ipynb` is the only preload component that performs cumulative crop merging and Qdrant snapshot creation.
